# 12a. Spline GAM — Base / P2-Q (Herbal Supplements)

This notebook fits the prior-free spline GAM on the frozen Notebook 09 query-only winner pool. It represents Base/P2-Q and provides the additive-model benchmark against the raw S1-Q order.

The model is a query-balanced negative-sampling logistic reranker with independent univariate spline effects. Its six-feature primary registry contains rank percentile, within-query normalized retrieval score, structured query–item match, log pre-query item-review count, item-review recency, and candidate Brand availability. No same-user history feature or Brand-history association enters the model, and the previously-reviewed-item block remains diagnostic only.

The GAM family uses one maximum-depth fit per outer fold at candidate depth 1,000. The fitted scoring function is then evaluated on the fixed candidate prefixes at depths 100, 300, 500, 700, and 1,000; no depth-specific GAM refitting is performed. Five user-grouped outer folds are used. Fitting is restricted to target-present non-cold queries, while all 1,968 cases remain in the evaluation population.

Base produces native out-of-fold predictions for strict-cold cases and exports the score-and-rank artifacts reused by the other GAM conditions. The recorded runtime covers method-specific feature preparation, fitting, scoring, evaluation, and export; it does not support a scope-matched deployment-cost ranking against other rerankers.

The received file contains 19 executed code cells, 14 output-bearing code cells, and no stored errors. Its execution counts are order-preserving but non-contiguous: 1, 2, 3, and 5 through 20.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%pip install -q -U pyarrow tqdm scikit-learn scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 675.5/675.5 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 59.9 MB/s eta 0:00:00


## 1. Condition and Fixed GAM Contract

In [3]:
# ==== Imports ====
import gc
import hashlib
import json
import math
import os
import pickle
import random
import re
import time
import warnings
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from scipy import sparse
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import SplineTransformer
from tqdm.auto import tqdm

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

print("Libraries loaded.")
print("Random seed:", RANDOM_SEED)

Libraries loaded.
Random seed: 42


In [5]:
# ==== Config ====
NOTEBOOK_NAME = '12a_no_prior_rerank_gam_herbal.ipynb'
CATEGORY_ID = 'herbal'
CATEGORY_FOLDER = 'herbal_supplements'
CATEGORY_LABEL = 'Herbal Supplements'
CONDITION_KEY = 's2q'
CANDIDATE_POOL_ROLE = 'baseline_query_only'
CANDIDATE_MANIFEST_OUTPUT_KEY = 'query_only_winner_long'
USE_USER_PRIOR_FEATURES = False

PROJECT_ROOT = Path('/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements')
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROCESSED_ITEMS_DIR = PROJECT_ROOT / "data/processed/items"
USER_SAMPLING_DIR = PROJECT_ROOT / "data/processed/user_sampling"
STAGE1_CANDIDATE_MANIFEST_PATH = OUTPUTS_DIR / "stage1_candidate_pools" / f"stage1_candidate_pool_export_manifest_{CATEGORY_ID}.json"
OUT_DIR = OUTPUTS_DIR / "stage2_nonpersonalized_rerank/gam_no_prior"
SHARED_ALL_PRIOR_MODEL_DIR = OUTPUTS_DIR / "stage2_nonpersonalized_rerank/gam"
SHARED_ALL_PRIOR_MODEL_ROLE = 'not_applicable'
SHARED_ALL_PRIOR_SCHEMA_PATH = SHARED_ALL_PRIOR_MODEL_DIR / "shared_all_prior_feature_schema.json"
SHARED_ALL_PRIOR_CONTRACT_PATH = SHARED_ALL_PRIOR_MODEL_DIR / "shared_all_prior_gam_contract.json"
SHARED_ALL_PRIOR_FOLD_ASSIGNMENTS_PATH = SHARED_ALL_PRIOR_MODEL_DIR / "shared_all_prior_query_folds.parquet"

# Exact Notebook 02 / Notebook 03 contracts. No fallback path search is permitted.
ITEM_SCHEMA_PATH = PROCESSED_ITEMS_DIR / 'herbal_item_schema_full.parquet'
ITEM_FACETS_PATH = PROCESSED_ITEMS_DIR / 'items_facets_herbal.parquet'
RETRIEVAL_ARTIFACT_MANIFEST_PATH = PROCESSED_ITEMS_DIR / "retrieval_artifact_manifest_herbal.json"
PRIOR_HISTORY_PATH = USER_SAMPLING_DIR / 'herbal_user_prior_review_history.parquet'
TEMPORAL_ARTIFACT_DIR = PROJECT_ROOT / "data" / "processed" / "items"
ITEM_REVIEW_TIME_INDEX_PATH = TEMPORAL_ARTIFACT_DIR / "item_review_time_index_herbal.parquet"
TEMPORAL_ARTIFACT_MANIFEST_PATH = TEMPORAL_ARTIFACT_DIR / "temporal_artifact_manifest_herbal.json"
EXPECTED_SOURCE_SCHEMA_NAME = "herbal_item_schema.parquet"
EXPECTED_TEMPORAL_REVIEW_SOURCE_NAME = "reviews_Herbal_Supplements_W2_2019_2022.parquet"
EXPECTED_TEMPORAL_LEAKAGE_RULE = "review_timestamp_ms < target_timestamp_ms"
if PROJECT_ROOT.name != CATEGORY_FOLDER or PROJECT_ROOT.parent.name != "categories":
    raise RuntimeError(
        f"PROJECT_ROOT must point to the finalized {CATEGORY_FOLDER} category workspace: {PROJECT_ROOT}"
    )
expected_temporal_relpath = Path("data") / "processed" / "items"
for temporal_path in [ITEM_REVIEW_TIME_INDEX_PATH, TEMPORAL_ARTIFACT_MANIFEST_PATH]:
    try:
        temporal_relpath = temporal_path.parent.relative_to(PROJECT_ROOT)
    except ValueError as exc:
        raise RuntimeError(
            f"Notebook 04 temporal artifact must be inside PROJECT_ROOT: {temporal_path}"
        ) from exc
    if temporal_relpath != expected_temporal_relpath or CATEGORY_ID not in temporal_path.name:
        raise RuntimeError(
            f"Notebook 04 temporal artifact must be the same-category Herbal artifact: {temporal_path}"
        )

REGIME_ORDER = ["cold", "weak", "strong"]
PROFILE_ROLES = [
    "brand",
    "category_or_product_type",
    "form_texture",
    "ingredient_or_composition",
    "need_benefit_concern",
    "claim_constraint",
    "target_context",
    "sensory",
]
NONBRAND_PROFILE_ROLES = [role for role in PROFILE_ROLES if role != "brand"]

FIT_POOL_DEPTH = 1000
REPORT_POOL_DEPTHS = [100, 300, 500, 700, 1000]
EXPORT_RERANKED_TOPK = 100
EXPORT_FEATURE_TABLE = False
SMOKE_TEST = False
SMOKE_QUERY_LIMIT = 60


# One common GAM capacity/optimizer contract is shared by all four conditions; b02 and Full re-fit it natively on the personalized pool (matched contract, no shared-model loading in the executed record).
GAM_N_FOLDS = 5
GAM_N_KNOTS = 6
GAM_SPLINE_DEGREE = 3
GAM_MAX_ITER = 500
GAM_TOL = 1e-4
GAM_CLASS_WEIGHT = None
GAM_SOLVER = "lbfgs"
GAM_C = 2.0
GAM_SCORE_CHUNK_SIZE = 250_000
GAM_HARD_NEGATIVES_PER_QUERY = 32
GAM_STRATIFIED_NEGATIVES_PER_QUERY = 32
GAM_NEGATIVE_BINS = 8
GAM_EFFECT_GRID_POINTS = 31
GAM_RESUME_VALID_FOLDS = False
RECENCY_HALF_LIFE_DAYS = 365.0
PROFILE_HISTORY_SATURATION_ITEMS = 10.0

PRIMARY_REGISTRY_POLICY_VERSION = "benchmark_primary_novel_item_registry_v1"
SPECIFICATION_ROLE = "primary"
ENABLE_EXACT_ITEM_FAMILIARITY = False

# Candidate-side evidence shared by Base, RankP, and Full.
S2Q_FEATURES = [
    "candidate_rank_pct",
    "candidate_score_norm",
    "query_item_structured_match",
    "item_prequery_review_count_log1p",
    "item_last_review_gap_days",
    "candidate_brand_present",
]

# Same-user evidence permitted in the primary next-novel-item specification.
PRIMARY_PRIOR_FEATURES = [
    "user_profile_affinity",
    "user_ingredient_or_composition_affinity",
    "user_need_benefit_concern_affinity",
    "profile_affinity_history_weighted",
    "profile_affinity_concentrated",
    "candidate_brand_prior_share",
    "candidate_brand_prior_interaction_count_log1p",
    "candidate_brand_recency_weight",
    "candidate_brand_share_x_brand_concentration",
]
EXACT_ITEM_DIAGNOSTIC_FEATURES = [
    "user_item_seen",
    "user_item_recency_strength",
]
EXACT_ITEM_FAMILIARITY_FEATURES = list(EXACT_ITEM_DIAGNOSTIC_FEATURES)
SHARED_ALL_PRIOR_FEATURES = [*S2Q_FEATURES, *PRIMARY_PRIOR_FEATURES]

EXPECTED_S2Q_FEATURE_COUNT = 6
EXPECTED_PRIOR_FEATURE_COUNT = 9
EXPECTED_ALL_PRIOR_FEATURE_COUNT = 15

if len(S2Q_FEATURES) != EXPECTED_S2Q_FEATURE_COUNT:
    raise RuntimeError("Primary GAM Base registry must contain exactly six features.")
if len(PRIMARY_PRIOR_FEATURES) != EXPECTED_PRIOR_FEATURE_COUNT:
    raise RuntimeError("Primary GAM prior block must contain exactly nine features.")
if len(SHARED_ALL_PRIOR_FEATURES) != EXPECTED_ALL_PRIOR_FEATURE_COUNT:
    raise RuntimeError("Primary GAM RankP/Full registry must contain exactly fifteen features.")
if set(SHARED_ALL_PRIOR_FEATURES) & set(EXACT_ITEM_DIAGNOSTIC_FEATURES):
    raise RuntimeError("Previously-reviewed-item features entered the primary GAM registry.")

BRAND_HISTORY_FEATURES = [
    "candidate_brand_prior_share",
    "candidate_brand_prior_interaction_count_log1p",
    "candidate_brand_recency_weight",
    "candidate_brand_share_x_brand_concentration",
]
QUERY_CONSTANT_DIAGNOSTIC_FEATURES = [
    "prior_event_count_log1p",
    "prior_unique_item_count_log1p",
    "history_item_count_log1p",
    "profile_entropy_mean",
    "user_last_interaction_gap_days",
    "user_prior_brand_entropy_norm",
]
BASE_FEATURES = list(S2Q_FEATURES)
PRIOR_FEATURES = [feature for feature in SHARED_ALL_PRIOR_FEATURES if feature not in S2Q_FEATURES]

# The same six candidate-common features receive the same spline encoding
# in Base and RankP. This prevents the feature ablation from being confounded by
# a different non-prior design matrix.
BASE_SPLINE_FEATURES = list(S2Q_FEATURES)
BASE_LINEAR_FEATURES = []
PRIOR_SPLINE_FEATURES = [] if CONDITION_KEY == "s2q" else list(PRIOR_FEATURES)
PRIOR_LINEAR_FEATURES = []

BASELINE_RERANK_METHOD = "stage1_baseline"
GAM_OUTPUT_RERANK_METHOD = 'gam_no_prior_rerank'
S2Q_OUT_DIR = OUTPUTS_DIR / "stage2_nonpersonalized_rerank/gam_no_prior"
S2Q_COLD_PREDICTIONS_PATH = S2Q_OUT_DIR / "s2q_cold_oof_predictions.parquet"
S2Q_COLD_TARGET_RANKS_PATH = S2Q_OUT_DIR / "s2q_cold_target_ranks_by_depth.parquet"
S2Q_FOLD_MODEL_DIR = S2Q_OUT_DIR / "fold_models_matched_non_cold"
S2Q_MODEL_CONTRACT_PATH = S2Q_FOLD_MODEL_DIR / "s2q_gam_model_contract.json"
expected_s2q_relpath = Path("outputs") / "stage2_nonpersonalized_rerank" / "gam_no_prior"
try:
    s2q_relpath = S2Q_OUT_DIR.relative_to(PROJECT_ROOT)
except ValueError as exc:
    raise RuntimeError(f"Base GAM artifact directory must be inside PROJECT_ROOT: {S2Q_OUT_DIR}") from exc
if s2q_relpath != expected_s2q_relpath:
    raise RuntimeError(f"Unexpected same-category Base GAM artifact directory: {S2Q_OUT_DIR}")
COMMON_RERANK_CONTRACT_DIR = OUTPUTS_DIR / "stage2_common/reranker_contracts"
COMMON_QUERY_FOLD_ASSIGNMENTS_PATH = COMMON_RERANK_CONTRACT_DIR / "category_query_folds_5.parquet"
COMMON_QUERY_FOLD_MANIFEST_PATH = COMMON_RERANK_CONTRACT_DIR / "category_query_folds_5_manifest.json"
COMMON_GAM_TRAINING_MEMBERSHIP_PATH = COMMON_RERANK_CONTRACT_DIR / "gam_baseline_training_sample_membership.parquet"
COMMON_GAM_TRAINING_MEMBERSHIP_MANIFEST_PATH = COMMON_RERANK_CONTRACT_DIR / "gam_baseline_training_sample_membership_manifest.json"
RUNTIME_METHOD_COMPONENTS_PATH = OUT_DIR / 'runtime_method_components_at1000_herbal.csv'
ALL_PRIOR_DEFINITION = "all_strict_pre_target_interactions_target_removed"
TRAINING_NEGATIVE_POLICY = "one_positive_plus_top32_hard_plus32_rank_stratified_negatives_query_balanced_half_mass"

ITEM_CACHE_VERSION = "gam_common_schema_item_catalog_v4"
BASE_FEATURE_CACHE_VERSION = "gam_common_schema_base_features_v7_temporal"
PROFILE_CACHE_VERSION = "gam_all_prior_profiles_v7_matched"
PRIOR_FEATURE_CACHE_VERSION = "gam_candidate_prior_features_v7_additive_interactions"
GAM_FEATURE_CONTRACT_VERSION = "gam_common_brand_contract_v8_primary_6_15"

CACHE_DIR = OUTPUTS_DIR / "stage2_feature_cache/gam_lite"
ITEM_CACHE_PATH = CACHE_DIR / f"item_catalog_lite_{CATEGORY_ID}.parquet"
ITEM_CACHE_MANIFEST_PATH = CACHE_DIR / f"item_catalog_lite_manifest_{CATEGORY_ID}.json"
BASE_FEATURE_CACHE_PATH = CACHE_DIR / f"base_features_{CANDIDATE_POOL_ROLE}_{CATEGORY_ID}.parquet"
BASE_FEATURE_CACHE_MANIFEST_PATH = CACHE_DIR / f"base_features_{CANDIDATE_POOL_ROLE}_manifest_{CATEGORY_ID}.json"
PROFILE_CACHE_PATH = CACHE_DIR / f"all_prior_profiles_{CATEGORY_ID}.parquet"
PROFILE_CACHE_MANIFEST_PATH = CACHE_DIR / f"all_prior_profiles_manifest_{CATEGORY_ID}.json"
PRIOR_FEATURE_CACHE_PATH = CACHE_DIR / f"prior_features_{CANDIDATE_POOL_ROLE}_{CATEGORY_ID}.parquet"
PRIOR_FEATURE_CACHE_MANIFEST_PATH = CACHE_DIR / f"prior_features_{CANDIDATE_POOL_ROLE}_manifest_{CATEGORY_ID}.json"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Notebook:", NOTEBOOK_NAME)
print("Category:", CATEGORY_LABEL)
print("Condition:", CONDITION_KEY)
print("Candidate role:", CANDIDATE_POOL_ROLE)
print("Shared All-Prior model role:", SHARED_ALL_PRIOR_MODEL_ROLE)
print("Output directory:", OUT_DIR)


Notebook: 12a_no_prior_rerank_gam_herbal.ipynb
Category: Herbal Supplements
Condition: s2q
Candidate role: baseline_query_only
Shared All-Prior model role: not_applicable
Output directory: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/gam_no_prior


In [6]:
# ==== Helpers ====
TOKEN_PATTERN = re.compile(r"[a-z0-9]+")


def normalize_space(value) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return re.sub(r"\s+", " ", str(value)).strip()


def normalize_label(value) -> str:
    text = normalize_space(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def tokenize_set(value) -> frozenset[str]:
    return frozenset(token for token in TOKEN_PATTERN.findall(normalize_label(value)) if len(token) > 1)


def stable_hash_text(value: str) -> str:
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def stable_hash_json(value) -> str:
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return stable_hash_text(payload)


def stable_frame_hash(frame: pd.DataFrame, columns: list[str], sort_columns: list[str]) -> str:
    work = frame.loc[:, columns].sort_values(sort_columns, kind="mergesort").reset_index(drop=True)
    payload = work.astype(str).agg("\x1f".join, axis=1).str.cat(sep="\n")
    return stable_hash_text(payload)


def file_signature(path_obj) -> dict:
    path = Path(path_obj)
    stat = path.stat()
    return {"path": str(path), "size": int(stat.st_size), "mtime_ns": int(stat.st_mtime_ns)}


def file_sha256(path_obj, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path_obj).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path_obj) -> dict:
    return json.loads(Path(path_obj).read_text(encoding="utf-8-sig"))


def write_json(path_obj, payload) -> None:
    path = Path(path_obj)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def require_columns(df: pd.DataFrame, columns: list[str], label: str) -> None:
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def as_bool_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0).ne(0)
    normalized = series.fillna("").astype(str).str.strip().str.lower()
    return normalized.isin({"1", "true", "t", "yes", "y"})


def normalize_regime(value) -> str:
    text = normalize_label(value).replace(" ", "_")
    aliases = {"no_history": "cold", "new": "cold", "low": "weak", "high": "strong"}
    normalized = aliases.get(text, text)
    if CATEGORY_ID == "herbal" and normalized and normalized not in set(REGIME_ORDER):
        raise RuntimeError(f"Unexpected Herbal regime label: {normalized!r}")
    return normalized


def to_timestamp_ms(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    positive = values[values.gt(0)]
    if not positive.empty and float(positive.median()) < 1e11:
        values = values * 1000.0
    return values


def normalized_entropy(counter: Counter) -> float:
    values = np.asarray([float(v) for v in counter.values() if float(v) > 0.0], dtype=float)
    if values.size <= 1:
        return 0.0
    probs = values / values.sum()
    entropy = float(-(probs * np.log(probs)).sum())
    return float(entropy / math.log(float(values.size)))


RUNTIME_ROWS = []
NOTEBOOK_TIMER_START = time.perf_counter()


@contextmanager
def runtime_step(step_name: str, **metadata):
    started = time.perf_counter()
    error = ""
    try:
        yield
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        raise
    finally:
        row = {"step_name": step_name, "runtime_sec": float(time.perf_counter() - started), "error": error}
        row.update(metadata)
        RUNTIME_ROWS.append(row)


## 2. Load the Frozen S1-Q Candidate Pool

In [7]:
# ==== Notebook 09 Candidate Contract ====
with runtime_step("load_stage1_candidates"):
    if not STAGE1_CANDIDATE_MANIFEST_PATH.exists():
        raise FileNotFoundError(f"Missing Notebook 09 manifest: {STAGE1_CANDIDATE_MANIFEST_PATH}")
    stage1_manifest = load_json(STAGE1_CANDIDATE_MANIFEST_PATH)
    if stage1_manifest.get("category_id") != CATEGORY_ID:
        raise RuntimeError("Notebook 09 category mismatch.")
    if stage1_manifest.get("candidate_budget_policy") != "exact_k_all_methods":
        raise RuntimeError("Notebook 09 must use the exact-K candidate policy.")
    if stage1_manifest.get("variable_candidate_count_allowed") is not False:
        raise RuntimeError("Variable candidate counts are not allowed.")

    output_paths_stage1 = stage1_manifest.get("output_paths", {})
    candidate_path = Path(output_paths_stage1.get(CANDIDATE_MANIFEST_OUTPUT_KEY, ""))
    if not candidate_path.exists():
        raise FileNotFoundError(f"Missing selected Notebook 09 candidate pool: {candidate_path}")

    if CANDIDATE_POOL_ROLE == "baseline_query_only":
        candidate_method_key = normalize_space(stage1_manifest.get("baseline_retrieval_winner_method_key"))
        candidate_method_label = normalize_space(stage1_manifest.get("baseline_retrieval_winner_method_label"))
        source_baseline_method_key = candidate_method_key
        source_baseline_method_label = candidate_method_label
    else:
        candidate_method_key = normalize_space(stage1_manifest.get("selected_personalized_method_slug"))
        candidate_method_label = normalize_space(stage1_manifest.get("selected_personalized_method_label"))
        source_baseline_method_key = normalize_space(stage1_manifest.get("baseline_retrieval_winner_method_key"))
        source_baseline_method_label = normalize_space(stage1_manifest.get("baseline_retrieval_winner_method_label"))
    if not candidate_method_key or not candidate_method_label:
        raise RuntimeError("Notebook 09 winner method identity is empty.")

    raw = pd.read_parquet(candidate_path)
    required = [
        "category_id", "case_id", "query_id", "user_id", "regime", "sampling_bracket",
        "target_selection_mode", "target_parent_asin", "gt_item_id", "target_timestamp_ms",
        "prior_history_n", "query_text", "candidate_pool_role", "method_slug",
        "retrieval_method", "candidate_parent_asin", "candidate_rank", "candidate_score",
        "candidate_score_source", "is_gt",
    ]
    require_columns(raw, required, "Notebook 09 candidate pool")
    if set(raw["category_id"].astype(str)) != {CATEGORY_ID}:
        raise RuntimeError("Candidate category_id does not match this notebook.")
    if set(raw["candidate_pool_role"].astype(str)) != {CANDIDATE_POOL_ROLE}:
        raise RuntimeError("Notebook 09 candidate_pool_role mismatch.")
    if set(raw["method_slug"].astype(str)) != {candidate_method_key}:
        raise RuntimeError("Notebook 09 method_slug mismatch.")
    if set(raw["retrieval_method"].astype(str)) != {candidate_method_label}:
        raise RuntimeError("Notebook 09 retrieval_method mismatch.")
    if not raw["target_parent_asin"].astype(str).eq(raw["gt_item_id"].astype(str)).all():
        raise RuntimeError("target_parent_asin and gt_item_id disagree.")

    candidate_df = pd.DataFrame({
        "case_id": raw["case_id"].astype(str),
        "query_id": raw["query_id"].astype(str),
        "user_id": raw["user_id"].fillna("").astype(str),
        "regime": raw["regime"].map(normalize_regime),
        "sampling_bracket": raw["sampling_bracket"].fillna("").astype(str),
        "target_selection_mode": raw["target_selection_mode"].fillna("").astype(str),
        "query_text": raw["query_text"].fillna("").astype(str),
        "target_item_id": raw["target_parent_asin"].astype(str),
        "target_timestamp_ms": to_timestamp_ms(raw["target_timestamp_ms"]).astype("int64"),
        "prior_history_n_upstream": pd.to_numeric(raw["prior_history_n"], errors="raise").astype(int),
        "candidate_item_id": raw["candidate_parent_asin"].astype(str),
        "candidate_rank": pd.to_numeric(raw["candidate_rank"], errors="raise").astype(int),
        "candidate_score": pd.to_numeric(raw["candidate_score"], errors="raise").astype(float),
        "candidate_score_source": raw["candidate_score_source"].fillna("").astype(str),
        "is_gt": as_bool_series(raw["is_gt"]).astype(np.int8),
    })
    if candidate_df["query_text"].str.strip().eq("").any():
        raise RuntimeError("Candidate rows contain empty query text.")
    if not np.isfinite(candidate_df["candidate_score"].to_numpy()).all():
        raise RuntimeError("Candidate scores contain non-finite values.")
    if candidate_df.duplicated(["query_id", "candidate_item_id"]).any():
        raise RuntimeError("Duplicate query-item candidates found.")
    if candidate_df.duplicated(["query_id", "candidate_rank"]).any():
        raise RuntimeError("Duplicate query-rank pairs found.")
    computed_gt = candidate_df["candidate_item_id"].eq(candidate_df["target_item_id"]).astype(np.int8)
    if not computed_gt.eq(candidate_df["is_gt"]).all():
        raise RuntimeError("is_gt is inconsistent with target identity.")
    if candidate_df.groupby("query_id")["is_gt"].sum().gt(1).any():
        raise RuntimeError("A query has more than one target candidate.")

    expected_k = int(stage1_manifest.get("effective_candidate_count_per_query", stage1_manifest.get("candidate_budget_k", 1000)))
    if expected_k != FIT_POOL_DEPTH:
        raise RuntimeError(f"Expected Notebook 09 candidate depth {FIT_POOL_DEPTH}, found {expected_k}.")
    count_qc = candidate_df.groupby("query_id").size()
    if not count_qc.eq(expected_k).all():
        raise RuntimeError(f"Exact-K candidate count failed: {count_qc.loc[~count_qc.eq(expected_k)].head().to_dict()}")
    rank_qc = candidate_df.groupby("query_id")["candidate_rank"].agg(["min", "max", "nunique"])
    if not ((rank_qc["min"] == 1).all() and (rank_qc["max"] == expected_k).all() and (rank_qc["nunique"] == expected_k).all()):
        raise RuntimeError("Candidate ranks are not exactly 1 through K.")

    query_meta_df = candidate_df[[
        "case_id", "query_id", "user_id", "regime", "sampling_bracket", "target_selection_mode",
        "query_text", "target_item_id", "target_timestamp_ms", "prior_history_n_upstream",
    ]].drop_duplicates("query_id").reset_index(drop=True)
    if query_meta_df["case_id"].duplicated().any():
        raise RuntimeError("Notebook 09 case_id must identify one evaluation query.")

    query_cache_path = Path(stage1_manifest.get("input_paths", {}).get("query_cache", ""))
    if not query_cache_path.exists():
        raise FileNotFoundError(f"Missing Notebook 06 query cache recorded by Notebook 09: {query_cache_path}")
    query_cache_df = pd.read_parquet(query_cache_path)
    require_columns(query_cache_df, ["case_id"], "Notebook 06 query cache")
    query_cache_df["case_id"] = query_cache_df["case_id"].astype(str)
    if query_cache_df["case_id"].duplicated().any():
        raise RuntimeError("Notebook 06 query cache contains duplicate case_id values.")
    query_cache_active = query_cache_df[query_cache_df["case_id"].isin(set(query_meta_df["case_id"]))].copy()

    if "brand_or_name_leak_flag" in query_cache_active.columns:
        brand_terms_added_to_synthetic_query_n = int(as_bool_series(query_cache_active["brand_or_name_leak_flag"]).sum())
        query_brand_leak_qc_source = "query_cache_brand_or_name_leak_flag"
    else:
        required_query_safety_audit_cols = [
            "query_evidence_source",
            "query_generation_status",
            "target_metadata_fallback_used",
            "item_context_fallback_used",
            "item_metadata_evidence_used",
            "historical_review_evidence_used",
            "user_prior_evidence_used",
            "rating_evidence_used",
            "sentiment_evidence_used",
            "insufficient_review_evidence",
        ]
        missing_query_safety_audit_cols = [
            col for col in required_query_safety_audit_cols
            if col not in query_cache_active.columns
        ]
        if missing_query_safety_audit_cols:
            raise RuntimeError(
                "Notebook 06 query cache is missing brand/name leak QC and required "
                f"target-review-safe audit columns: {missing_query_safety_audit_cols}"
            )
        forbidden_query_fallback_cols = [
            "target_metadata_fallback_used",
            "item_context_fallback_used",
            "item_metadata_evidence_used",
            "historical_review_evidence_used",
            "user_prior_evidence_used",
            "rating_evidence_used",
            "sentiment_evidence_used",
            "insufficient_review_evidence",
        ]
        target_review_safe_source = query_cache_active["query_evidence_source"].astype(str).eq("target_review_safe_signals_only")
        target_review_safe_status = query_cache_active["query_generation_status"].astype(str).eq("generated_from_review_safe_signals")
        no_forbidden_query_fallback = ~query_cache_active[forbidden_query_fallback_cols].apply(as_bool_series).any(axis=1)
        inferred_safe = target_review_safe_source & target_review_safe_status & no_forbidden_query_fallback
        if not inferred_safe.all():
            bad_query_safety_rows = query_cache_active.loc[~inferred_safe, ["case_id"] + required_query_safety_audit_cols].head(10).to_dict(orient="records")
            raise RuntimeError(
                "Notebook 06 query cache lacks brand_or_name_leak_flag and does not prove "
                f"target-review-safe query generation for all active cases: {bad_query_safety_rows}"
            )
        brand_terms_added_to_synthetic_query_n = 0
        query_brand_leak_qc_source = "inferred_zero_from_target_review_safe_query_cache_audit"

    if brand_terms_added_to_synthetic_query_n != 0:
        raise RuntimeError("Notebook 06 query cache reports brand/name leakage in the synthetic query.")

    if SMOKE_TEST:
        rng = np.random.default_rng(RANDOM_SEED)
        sampled_query_ids = rng.choice(query_meta_df["query_id"].to_numpy(), size=min(SMOKE_QUERY_LIMIT, len(query_meta_df)), replace=False)
        candidate_df = candidate_df[candidate_df["query_id"].isin(sampled_query_ids)].copy()
        query_meta_df = query_meta_df[query_meta_df["query_id"].isin(sampled_query_ids)].copy()

    query_set_hash = stable_hash_text("\n".join(sorted(query_meta_df["query_id"].astype(str))))
    candidate_identity_before_hash = stable_frame_hash(
        candidate_df, ["query_id", "candidate_item_id", "candidate_rank"], ["query_id", "candidate_rank", "candidate_item_id"]
    )
    candidate_source_signature = file_signature(candidate_path)
    stage1_manifest_hash = stable_hash_text(STAGE1_CANDIDATE_MANIFEST_PATH.read_text(encoding="utf-8-sig"))
    print("Candidate pool role:", CANDIDATE_POOL_ROLE)
    print("Candidate method:", candidate_method_key, "-", candidate_method_label)
    print("Queries:", query_meta_df["query_id"].nunique(), "Candidate rows:", len(candidate_df), "Exact K:", expected_k)


Candidate pool role: baseline_query_only
Candidate method: hybrid_dense_bm25 - Dense-BM25 Hybrid
Queries: 1968 Candidate rows: 1968000 Exact K: 1000


## 3. Load Candidate Metadata and Point-in-Time Evidence

In [8]:
# ==== Notebook 03 All Prior Contract ====
prior_events_df = pd.DataFrame(columns=["query_id", "case_id", "user_id", "item_id", "prior_timestamp_ms"])
prior_unique_df = prior_events_df.copy()
prior_event_count_map = {}
same_target_item_prior_rows = 0
temporal_validation_passed = True

if USE_USER_PRIOR_FEATURES:
    with runtime_step("load_all_prior"):
        if not PRIOR_HISTORY_PATH.exists():
            raise FileNotFoundError(f"Missing Notebook 03 strict prior-history artifact: {PRIOR_HISTORY_PATH}")
        prior_required = [
            "case_id", "user_id", "target_timestamp_ms",
            "prior_item_id", "prior_timestamp_ms",
        ]
        prior_item_column = "prior_item_id"
        prior_raw = pd.read_parquet(PRIOR_HISTORY_PATH, columns=prior_required)
        require_columns(prior_raw, prior_required, "Notebook 03 prior history")
        prior_std = pd.DataFrame({
            "case_id": prior_raw["case_id"].astype(str),
            "user_id": prior_raw["user_id"].fillna("").astype(str),
            "item_id": prior_raw[prior_item_column].fillna("").astype(str),
            "prior_timestamp_ms": to_timestamp_ms(prior_raw["prior_timestamp_ms"]),
            "prior_target_timestamp_ms": to_timestamp_ms(prior_raw["target_timestamp_ms"]),
        })
        query_lookup = query_meta_df[["case_id", "query_id", "user_id", "target_item_id", "target_timestamp_ms"]].copy()
        prior_std = prior_std.merge(query_lookup, on=["case_id", "user_id"], how="inner", validate="many_to_one")
        if not prior_std["prior_target_timestamp_ms"].astype("int64").eq(prior_std["target_timestamp_ms"].astype("int64")).all():
            raise RuntimeError("Notebook 03 and Notebook 09 target timestamps disagree.")
        prior_std = prior_std[
            prior_std["user_id"].ne("") & prior_std["item_id"].ne("") & prior_std["prior_timestamp_ms"].notna()
        ].copy()
        prior_std["prior_timestamp_ms"] = prior_std["prior_timestamp_ms"].astype("int64")
        temporal_validation_passed = bool(prior_std["prior_timestamp_ms"].lt(prior_std["target_timestamp_ms"]).all())
        if not temporal_validation_passed:
            raise RuntimeError("All Prior contains an interaction at or after the target timestamp.")
        same_target_item_prior_rows = int(prior_std["item_id"].eq(prior_std["target_item_id"]).sum())
        if same_target_item_prior_rows != 0:
            raise RuntimeError("All Prior contains the held-out target parent item.")
        prior_events_df = prior_std[["query_id", "case_id", "user_id", "item_id", "prior_timestamp_ms"]].reset_index(drop=True)
        prior_event_count_map = prior_events_df.groupby("query_id").size().astype(int).to_dict()
        prior_unique_df = (
            prior_events_df.sort_values(["query_id", "prior_timestamp_ms", "item_id"], kind="mergesort")
            .drop_duplicates(["query_id", "item_id"], keep="last")
            .reset_index(drop=True)
        )
        actual_prior_count = prior_events_df.groupby("query_id").size().reindex(query_meta_df["query_id"], fill_value=0).astype(int)
        regime_by_query = query_meta_df.set_index("query_id")["regime"].map(normalize_regime)
        regime_by_query.index = regime_by_query.index.astype(str)
        cold_mismatch = regime_by_query.eq("cold") != actual_prior_count.eq(0)
        if cold_mismatch.any():
            raise RuntimeError(
                "Cold regime and actual strict prior history disagree: "
                f"{cold_mismatch[cold_mismatch].index[:10].tolist()}"
            )
        print("All Prior events:", len(prior_events_df), "Unique prior query-item rows:", len(prior_unique_df))
else:
    print("Base: Notebook 03 history is not loaded.")


Base: Notebook 03 history is not loaded.


In [9]:
# ==== Exact Notebook 02 Common Item Schema ====
with runtime_step("load_or_build_item_cache"):
    if not ITEM_SCHEMA_PATH.exists() or not ITEM_FACETS_PATH.exists():
        raise FileNotFoundError(f"Missing Notebook 02 item artifacts: {ITEM_SCHEMA_PATH}, {ITEM_FACETS_PATH}")
    if not RETRIEVAL_ARTIFACT_MANIFEST_PATH.exists():
        raise FileNotFoundError(f"Missing current Notebook 04 retrieval manifest: {RETRIEVAL_ARTIFACT_MANIFEST_PATH}")
    retrieval_manifest = load_json(RETRIEVAL_ARTIFACT_MANIFEST_PATH)
    if Path(retrieval_manifest.get("schema_path", "")).name != EXPECTED_SOURCE_SCHEMA_NAME:
        raise RuntimeError(
            "Notebook 04 retrieval manifest points to an unexpected Notebook 02 schema: "
            f"{retrieval_manifest.get('schema_path')}"
        )

    # Notebook 02 item schema provides the canonical candidate-visible brand field.
    schema_required = ["parent_asin", "brand_facet_text", "common_query_safe_facet_text"]
    item_schema = pd.read_parquet(ITEM_SCHEMA_PATH, columns=schema_required)
    require_columns(item_schema, schema_required, "Notebook 02 item schema")
    row_counts = retrieval_manifest.get("row_counts", {})
    item_docs_row_counts = retrieval_manifest.get("item_docs_row_counts", {})

    expected_active_rows = row_counts.get(
        "active_item_docs",
        item_docs_row_counts.get("item_docs", row_counts.get("item_docs", -1)),
    )
    expected_active_rows = int(expected_active_rows)

    if expected_active_rows <= 0 or expected_active_rows != len(item_schema):
        raise RuntimeError(
            "Notebook 02 active item schema row count disagrees with the current Notebook 04 manifest: "
            f"schema={len(item_schema)}, manifest={expected_active_rows}, "
            f"available_row_count_keys={sorted(row_counts.keys())}, "
            f"available_item_docs_row_count_keys={sorted(item_docs_row_counts.keys())}"
        )
    item_schema["item_id"] = item_schema["parent_asin"].astype(str)
    if item_schema["item_id"].duplicated().any():
        raise RuntimeError("Notebook 02 item schema must be unique by parent_asin.")
    item_schema["brand_norm"] = item_schema["brand_facet_text"].fillna("").map(normalize_label)
    item_schema["item_query_safe_text"] = item_schema["common_query_safe_facet_text"].fillna("").astype(str)

    facet_required = ["parent_asin", "facet_role", "facet_value_norm", "is_brand", "is_review_derived", "is_profile_safe"]
    item_facets = pd.read_parquet(ITEM_FACETS_PATH, columns=facet_required)
    require_columns(item_facets, facet_required, "Notebook 02 common long facets")
    item_facets["item_id"] = item_facets["parent_asin"].astype(str)
    item_facets["facet_role"] = item_facets["facet_role"].astype(str)
    item_facets["facet_value_norm"] = item_facets["facet_value_norm"].fillna("").map(normalize_label)
    unknown_roles = sorted(set(item_facets["facet_role"]) - set(PROFILE_ROLES) - {"review_derived_signal"})
    if unknown_roles:
        raise RuntimeError(f"Unexpected Notebook 02 facet roles: {unknown_roles}")
    profile_facets = item_facets[
        as_bool_series(item_facets["is_profile_safe"])
        & ~as_bool_series(item_facets["is_review_derived"])
        & item_facets["facet_role"].isin(PROFILE_ROLES)
        & item_facets["facet_value_norm"].ne("")
    ][["item_id", "facet_role", "facet_value_norm", "is_brand"]].drop_duplicates()
    if as_bool_series(profile_facets.loc[profile_facets["facet_role"].ne("brand"), "is_brand"]).any():
        raise RuntimeError("Notebook 02 is_brand flag conflicts with facet_role.")

    family_map = defaultdict(lambda: defaultdict(list))
    for row in profile_facets.itertuples(index=False):
        family_map[str(row.item_id)][str(row.facet_role)].append(str(row.facet_value_norm))
    item_family_map = {
        item_id: {role: sorted(set(values)) for role, values in role_values.items()}
        for item_id, role_values in family_map.items()
    }
    item_brand_map = item_schema.set_index("item_id")["brand_norm"].to_dict()
    item_text_map = item_schema.set_index("item_id")["item_query_safe_text"].to_dict()

    candidate_item_ids = set(candidate_df["candidate_item_id"].astype(str))
    prior_item_ids = set(prior_events_df["item_id"].astype(str)) if USE_USER_PRIOR_FEATURES else set()
    missing_items = sorted((candidate_item_ids | prior_item_ids) - set(item_schema["item_id"].astype(str)))
    if missing_items:
        raise RuntimeError(f"Notebook 02 item schema misses required items: n={len(missing_items)}")
    if USE_USER_PRIOR_FEATURES:
        # Schema coverage may remove unusable prior events. Recheck the effective
        # history policy after that removal so a nominally non-cold query cannot
        # silently become a zero-prior training case.
        effective_prior_count = (
            prior_events_df.groupby("query_id").size()
            .reindex(query_meta_df["query_id"].astype(str), fill_value=0)
            .astype(int)
        )
        effective_regime = query_meta_df.set_index("query_id")["regime"].map(normalize_regime)
        effective_regime.index = effective_regime.index.astype(str)
        effective_cold_mismatch = effective_regime.eq("cold") != effective_prior_count.eq(0)
        if effective_cold_mismatch.any():
            raise RuntimeError(
                "Effective prior history after Notebook 02 schema coverage no longer matches the cold regime: "
                f"{effective_cold_mismatch[effective_cold_mismatch].index[:10].tolist()}"
            )

    candidate_df["candidate_brand_raw"] = candidate_df["candidate_item_id"].map(item_brand_map).fillna("").astype(str)
    candidate_brand_nonnull_rate = float(candidate_df["candidate_brand_raw"].str.strip().ne("").mean())

    item_cache_manifest = {
        "cache_version": ITEM_CACHE_VERSION,
        "category_id": CATEGORY_ID,
        "item_schema_signature": file_signature(ITEM_SCHEMA_PATH),
        "item_facets_signature": file_signature(ITEM_FACETS_PATH),
        "brand_column": "brand_facet_text",
        "query_safe_text_column": "common_query_safe_facet_text",
        "profile_roles": PROFILE_ROLES,
        "historical_population_review_signals_in_user_profile": False,
    }
    print("Candidate brand non-null rate:", round(candidate_brand_nonnull_rate, 6))


Candidate brand non-null rate: 0.997478


## 4. Construct the Six-Feature Base Registry

In [10]:
# ==== Shared Candidate-side Features ====
def _load_current_temporal_manifest() -> dict:
    if not TEMPORAL_ARTIFACT_MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"Missing current Notebook 04 temporal manifest: {TEMPORAL_ARTIFACT_MANIFEST_PATH}"
        )

    manifest = load_json(TEMPORAL_ARTIFACT_MANIFEST_PATH)

    review_source_path = manifest.get(
        "review_source_path",
        manifest.get("review_metadata_source", ""),
    )
    if Path(review_source_path).name != EXPECTED_TEMPORAL_REVIEW_SOURCE_NAME:
        raise RuntimeError(
            "Notebook 04 temporal manifest points to an unexpected raw review source: "
            f"{review_source_path}"
        )

    temporal_rule = manifest.get(
        "temporal_review_strict_rule",
        manifest.get("leakage_rule"),
    )
    if temporal_rule != EXPECTED_TEMPORAL_LEAKAGE_RULE:
        raise RuntimeError(
            "Notebook 04 temporal leakage rule mismatch: "
            f"{temporal_rule}"
        )

    return manifest


def _base_cache_contract() -> dict:
    if not ITEM_REVIEW_TIME_INDEX_PATH.exists():
        raise FileNotFoundError(
            f"Missing category-specific Notebook 04 temporal index: {ITEM_REVIEW_TIME_INDEX_PATH}"
        )
    temporal_manifest = _load_current_temporal_manifest()
    return {
        "cache_version": BASE_FEATURE_CACHE_VERSION,
        "category_id": CATEGORY_ID,
        "candidate_pool_role": CANDIDATE_POOL_ROLE,
        "candidate_source_signature": candidate_source_signature,
        "stage1_manifest_hash": stage1_manifest_hash,
        "query_set_hash": query_set_hash,
        "feature_names": BASE_FEATURES,
        "temporal_source_signature": file_signature(ITEM_REVIEW_TIME_INDEX_PATH),
        "brand_column": "brand_facet_text",
        "structured_match_roles": NONBRAND_PROFILE_ROLES,
        "strict_prequery_temporal_features": True,
    }


def _build_base_features() -> pd.DataFrame:
    work = candidate_df.copy()
    work["candidate_rank_pct"] = (
        (work["candidate_rank"] - 1) / max(expected_k - 1, 1)
    ).astype(np.float32)
    score_min = work.groupby("query_id")["candidate_score"].transform("min")
    score_max = work.groupby("query_id")["candidate_score"].transform("max")
    score_span = score_max - score_min
    rank_fallback = 1.0 - work["candidate_rank_pct"]
    work["candidate_score_norm"] = np.where(
        score_span.gt(1e-12),
        (work["candidate_score"] - score_min) / score_span,
        rank_fallback,
    ).astype(np.float32)

    query_token_map = {
        str(row.query_id): tokenize_set(row.query_text)
        for row in query_meta_df[["query_id", "query_text"]].itertuples(index=False)
    }
    item_structured_token_map = {
        str(item_id): frozenset(
            token
            for role in NONBRAND_PROFILE_ROLES
            for label in family_values.get(role, [])
            for token in tokenize_set(label)
        )
        for item_id, family_values in item_family_map.items()
    }
    query_ids = work["query_id"].astype(str).to_numpy()
    item_ids = work["candidate_item_id"].astype(str).to_numpy()
    work["query_item_structured_match"] = np.fromiter(
        (
            len(
                query_token_map.get(query_id, frozenset())
                & item_structured_token_map.get(item_id, frozenset())
            )
            / max(len(query_token_map.get(query_id, frozenset())), 1)
            for query_id, item_id in zip(query_ids, item_ids)
        ),
        dtype=np.float32,
        count=len(work),
    )
    work["candidate_brand_present"] = (
        work["candidate_brand_raw"].str.strip().ne("").astype(np.float32)
    )

    if not ITEM_REVIEW_TIME_INDEX_PATH.exists():
        raise FileNotFoundError(
            f"Missing category-specific Notebook 04 temporal index: {ITEM_REVIEW_TIME_INDEX_PATH}"
        )
    temporal_manifest = _load_current_temporal_manifest()
    item_time_df = pd.read_parquet(
        ITEM_REVIEW_TIME_INDEX_PATH,
        columns=["parent_asin", "review_timestamp_ms"],
    )
    item_time_df["parent_asin"] = (
        item_time_df["parent_asin"].fillna("").astype(str).str.strip()
    )
    item_time_df["review_timestamp_ms"] = pd.to_numeric(
        item_time_df["review_timestamp_ms"], errors="coerce"
    )
    item_time_df = item_time_df.dropna(subset=["review_timestamp_ms"])
    item_time_df["review_timestamp_ms"] = item_time_df["review_timestamp_ms"].astype(np.int64)
    expected_temporal_rows = int(
        temporal_manifest.get("row_counts", {}).get(
            "item_review_time_index",
            temporal_manifest.get("row_counts", {}).get("item_review_timestamp_index", -1),
        )
    )
    if expected_temporal_rows <= 0 or expected_temporal_rows != len(item_time_df):
        raise RuntimeError(
            "Notebook 04 temporal index row count disagrees with its manifest: "
            f"index={len(item_time_df)}, manifest={expected_temporal_rows}"
        )
    item_timestamp_map = {
        str(item_id): np.sort(group["review_timestamp_ms"].to_numpy(dtype=np.int64))
        for item_id, group in item_time_df.groupby("parent_asin", sort=False)
        if str(item_id) and len(group)
    }
    prequery_count = np.zeros(len(work), dtype=np.float32)
    last_gap_days = np.full(len(work), 3650.0, dtype=np.float32)
    target_times = pd.to_numeric(
        work["target_timestamp_ms"], errors="raise"
    ).astype(np.int64).to_numpy()
    for index, (item_id, target_ts) in enumerate(zip(item_ids, target_times)):
        timestamps = item_timestamp_map.get(str(item_id))
        if timestamps is None or len(timestamps) == 0:
            continue
        # side="left" enforces review_timestamp_ms < target_timestamp_ms.
        right = int(np.searchsorted(timestamps, int(target_ts), side="left"))
        prequery_count[index] = float(right)
        if right > 0:
            last_gap_days[index] = float(
                max(int(target_ts) - int(timestamps[right - 1]), 0) / 86_400_000.0
            )
    work["item_prequery_review_count_log1p"] = np.log1p(prequery_count).astype(np.float32)
    work["item_last_review_gap_days"] = np.clip(
        last_gap_days, 0.0, 3650.0
    ).astype(np.float32)
    return work


with runtime_step("load_or_build_base_features"):
    expected_base_contract = _base_cache_contract()
    if (
        BASE_FEATURE_CACHE_PATH.exists()
        and BASE_FEATURE_CACHE_MANIFEST_PATH.exists()
        and load_json(BASE_FEATURE_CACHE_MANIFEST_PATH) == expected_base_contract
    ):
        feature_df = pd.read_parquet(BASE_FEATURE_CACHE_PATH)
    else:
        feature_df = _build_base_features()
        BASE_FEATURE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        feature_df.to_parquet(BASE_FEATURE_CACHE_PATH, index=False)
        write_json(BASE_FEATURE_CACHE_MANIFEST_PATH, expected_base_contract)
    derived_gt_in_pool = (
        feature_df.groupby("query_id")["is_gt"]
        .transform("max")
        .astype(np.int8)
    )
    if "gt_in_pool" in feature_df.columns:
        observed_gt_in_pool = pd.to_numeric(
            feature_df["gt_in_pool"], errors="raise"
        ).astype(np.int8)
        if not observed_gt_in_pool.eq(derived_gt_in_pool).all():
            raise RuntimeError("Cached gt_in_pool disagrees with query-level is_gt maxima.")
    feature_df["gt_in_pool"] = derived_gt_in_pool
    require_columns(
        feature_df,
        [*BASE_FEATURES, "candidate_brand_raw", "query_id", "candidate_item_id", "is_gt", "gt_in_pool"],
        "base feature table",
    )
    candidate_identity_after_base_hash = stable_frame_hash(
        feature_df,
        ["query_id", "candidate_item_id", "candidate_rank"],
        ["query_id", "candidate_rank", "candidate_item_id"],
    )
    if (
        candidate_identity_after_base_hash != candidate_identity_before_hash
        or len(feature_df) != len(candidate_df)
    ):
        raise RuntimeError("Candidate rows or identities changed during base feature construction.")


In [11]:
# ==== Shared All Prior Profiles and Candidate Features ====
def _build_profile_cache() -> pd.DataFrame:
    rows = []
    prior_groups = {query_id: group for query_id, group in prior_events_df.groupby("query_id", sort=False)}
    empty_group = pd.DataFrame(columns=prior_events_df.columns)
    for query_row in tqdm(query_meta_df.itertuples(index=False), total=len(query_meta_df), desc="Build All Prior profiles"):
        query_id = str(query_row.query_id)
        group = prior_groups.get(query_id, empty_group)
        family_counters = {role: Counter() for role in NONBRAND_PROFILE_ROLES}
        brand_counter = Counter()
        brand_unique_items = defaultdict(set)
        brand_last_ts = {}
        seen_items = set()
        item_last_ts = {}
        for prior_row in group.itertuples(index=False):
            item_id = str(prior_row.item_id)
            prior_ts = int(prior_row.prior_timestamp_ms)
            seen_items.add(item_id)
            item_last_ts[item_id] = max(prior_ts, item_last_ts.get(item_id, -1))
            brand = item_brand_map.get(item_id, "")
            if brand:
                brand_counter[brand] += 1
                brand_unique_items[brand].add(item_id)
                brand_last_ts[brand] = max(prior_ts, brand_last_ts.get(brand, -1))
            family_labels = item_family_map.get(item_id, {})
            for role in NONBRAND_PROFILE_ROLES:
                for label in family_labels.get(role, []):
                    family_counters[role][label] += 1

        family_prob_payload = {
            role: {label: float(count / max(sum(counter.values()), 1)) for label, count in counter.items()}
            for role, counter in family_counters.items()
        }
        entropy_values = [normalized_entropy(counter) for counter in family_counters.values() if sum(counter.values()) > 0]
        profile_entropy_mean = float(np.mean(entropy_values)) if entropy_values else 0.0
        prior_event_count_log1p = float(np.log1p(len(group)))
        prior_unique_item_count_log1p = float(np.log1p(len(seen_items)))
        # Backward-compatible diagnostic alias; it is unique-item breadth, not event volume.
        history_item_count_log1p = prior_unique_item_count_log1p
        history_strength = float(
            min(prior_unique_item_count_log1p / max(np.log1p(PROFILE_HISTORY_SATURATION_ITEMS), 1e-12), 1.0)
        )
        last_prior_timestamp = int(group["prior_timestamp_ms"].max()) if len(group) else None
        target_timestamp = int(getattr(query_row, "target_timestamp_ms"))
        user_last_interaction_gap_days = (
            float(np.clip((target_timestamp - last_prior_timestamp) / 86_400_000.0, 0.0, 3650.0))
            if last_prior_timestamp is not None else 3650.0
        )
        max_brand_count = max(brand_counter.values(), default=0)
        dominant_brands = sorted([brand for brand, count in brand_counter.items() if count == max_brand_count and count > 0])
        rows.append({
            "query_id": query_id,
            "profile_history_event_n": int(len(group)),
            "profile_history_item_n": int(len(seen_items)),
            "prior_event_count_log1p": prior_event_count_log1p,
            "prior_unique_item_count_log1p": prior_unique_item_count_log1p,
            "history_item_count_log1p": history_item_count_log1p,
            "user_last_interaction_gap_days": user_last_interaction_gap_days,
            "history_strength": history_strength,
            "profile_entropy_mean": profile_entropy_mean,
            "profile_concentration": float(max(0.0, 1.0 - profile_entropy_mean)),
            "seen_items_json": json.dumps(sorted(seen_items), separators=(",", ":")),
            "item_last_ts_json": json.dumps(item_last_ts, separators=(",", ":")),
            "family_prob_json": json.dumps(family_prob_payload, ensure_ascii=False, separators=(",", ":")),
            "brand_count_json": json.dumps(dict(brand_counter), ensure_ascii=False, separators=(",", ":")),
            "brand_unique_item_count_json": json.dumps({k: len(v) for k, v in brand_unique_items.items()}, ensure_ascii=False, separators=(",", ":")),
            "brand_last_ts_json": json.dumps(brand_last_ts, ensure_ascii=False, separators=(",", ":")),
            "dominant_brands_json": json.dumps(dominant_brands, ensure_ascii=False, separators=(",", ":")),
            "user_prior_unique_brand_count": int(len(brand_counter)),
            "user_prior_brand_entropy_norm": float(normalized_entropy(brand_counter)),
        })
    return pd.DataFrame(rows)


def _profile_cache_contract() -> dict:
    return {
        "cache_version": PROFILE_CACHE_VERSION,
        "category_id": CATEGORY_ID,
        "query_set_hash": query_set_hash,
        "prior_source_signature": file_signature(PRIOR_HISTORY_PATH),
        "item_schema_signature": file_signature(ITEM_SCHEMA_PATH),
        "item_facets_signature": file_signature(ITEM_FACETS_PATH),
        "all_prior_definition": ALL_PRIOR_DEFINITION,
        "profile_roles": PROFILE_ROLES,
        "review_derived_roles_allowed": False,
    }


def _candidate_prior_cache_contract() -> dict:
    return {
        "cache_version": PRIOR_FEATURE_CACHE_VERSION,
        "category_id": CATEGORY_ID,
        "candidate_pool_role": CANDIDATE_POOL_ROLE,
        "candidate_source_signature": candidate_source_signature,
        "query_set_hash": query_set_hash,
        "profile_cache_version": PROFILE_CACHE_VERSION,
        "feature_names": PRIOR_FEATURES,
    }


if USE_USER_PRIOR_FEATURES:
    with runtime_step("load_or_build_all_prior_profiles"):
        expected_profile_contract = _profile_cache_contract()
        if PROFILE_CACHE_PATH.exists() and PROFILE_CACHE_MANIFEST_PATH.exists() and load_json(PROFILE_CACHE_MANIFEST_PATH) == expected_profile_contract:
            query_profiles_df = pd.read_parquet(PROFILE_CACHE_PATH)
        else:
            query_profiles_df = _build_profile_cache()
            PROFILE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
            query_profiles_df.to_parquet(PROFILE_CACHE_PATH, index=False)
            write_json(PROFILE_CACHE_MANIFEST_PATH, expected_profile_contract)

    with runtime_step("load_or_build_candidate_prior_features"):
        expected_prior_feature_contract = _candidate_prior_cache_contract()
        if PRIOR_FEATURE_CACHE_PATH.exists() and PRIOR_FEATURE_CACHE_MANIFEST_PATH.exists() and load_json(PRIOR_FEATURE_CACHE_MANIFEST_PATH) == expected_prior_feature_contract:
            feature_df = pd.read_parquet(PRIOR_FEATURE_CACHE_PATH)
        else:
            profile_map = query_profiles_df.set_index("query_id").to_dict(orient="index")
            seen_map = {qid: set(json.loads(v["seen_items_json"])) for qid, v in profile_map.items()}
            item_last_map = {qid: {str(k): int(ts) for k, ts in json.loads(v["item_last_ts_json"]).items()} for qid, v in profile_map.items()}
            family_prob_map = {qid: json.loads(v["family_prob_json"]) for qid, v in profile_map.items()}
            brand_count_map = {qid: {str(k): int(x) for k, x in json.loads(v["brand_count_json"]).items()} for qid, v in profile_map.items()}
            brand_unique_item_map = {qid: {str(k): int(x) for k, x in json.loads(v["brand_unique_item_count_json"]).items()} for qid, v in profile_map.items()}
            brand_last_map = {qid: {str(k): int(x) for k, x in json.loads(v["brand_last_ts_json"]).items()} for qid, v in profile_map.items()}
            dominant_brand_map = {qid: set(json.loads(v["dominant_brands_json"])) for qid, v in profile_map.items()}
            query_ids = feature_df["query_id"].astype(str).to_numpy()
            item_ids = feature_df["candidate_item_id"].astype(str).to_numpy()
            brands = feature_df["candidate_brand_raw"].astype(str).to_numpy()
            target_times = feature_df["target_timestamp_ms"].astype(np.int64).to_numpy()
            profile_index = query_profiles_df.set_index("query_id")
            feature_df["history_strength"] = feature_df["query_id"].map(profile_index["history_strength"]).fillna(0.0).astype(np.float32)
            feature_df["prior_event_count_log1p"] = feature_df["query_id"].map(profile_index["prior_event_count_log1p"]).fillna(0.0).astype(np.float32)
            feature_df["prior_unique_item_count_log1p"] = feature_df["query_id"].map(profile_index["prior_unique_item_count_log1p"]).fillna(0.0).astype(np.float32)
            feature_df["history_item_count_log1p"] = feature_df["prior_unique_item_count_log1p"].astype(np.float32)
            feature_df["profile_entropy_mean"] = feature_df["query_id"].map(profile_index["profile_entropy_mean"]).fillna(0.0).astype(np.float32)
            feature_df["profile_concentration"] = feature_df["query_id"].map(profile_index["profile_concentration"]).fillna(0.0).astype(np.float32)
            feature_df["user_last_interaction_gap_days"] = feature_df["query_id"].map(profile_index["user_last_interaction_gap_days"]).fillna(3650.0).astype(np.float32)

            def family_affinity(query_id: str, item_id: str, role: str) -> float:
                labels = item_family_map.get(item_id, {}).get(role, [])
                probabilities = family_prob_map.get(query_id, {}).get(role, {})
                return float(np.mean([float(probabilities.get(label, 0.0)) for label in labels])) if labels and probabilities else 0.0

            feature_df["user_item_seen"] = np.fromiter((item_id in seen_map.get(qid, set()) for qid, item_id in zip(query_ids, item_ids)), dtype=np.int8, count=len(feature_df))
            feature_df["user_item_recency_strength"] = np.fromiter((
                math.exp(-max((target_ts - item_last_map.get(qid, {}).get(item_id, 0)) / 86_400_000.0, 0.0) / RECENCY_HALF_LIFE_DAYS)
                if item_id in item_last_map.get(qid, {}) else 0.0
                for qid, item_id, target_ts in zip(query_ids, item_ids, target_times)
            ), dtype=np.float32, count=len(feature_df))
            feature_df["user_ingredient_or_composition_affinity"] = np.fromiter((family_affinity(qid, item_id, "ingredient_or_composition") for qid, item_id in zip(query_ids, item_ids)), dtype=np.float32, count=len(feature_df))
            feature_df["user_need_benefit_concern_affinity"] = np.fromiter((family_affinity(qid, item_id, "need_benefit_concern") for qid, item_id in zip(query_ids, item_ids)), dtype=np.float32, count=len(feature_df))
            feature_df["user_profile_affinity"] = np.fromiter((
                float(np.mean([family_affinity(qid, item_id, role) for role in NONBRAND_PROFILE_ROLES]))
                for qid, item_id in zip(query_ids, item_ids)
            ), dtype=np.float32, count=len(feature_df))
            feature_df["profile_affinity_concentrated"] = (feature_df["user_profile_affinity"] * feature_df["profile_concentration"]).astype(np.float32)
            feature_df["profile_affinity_history_weighted"] = (feature_df["user_profile_affinity"] * feature_df["history_strength"]).astype(np.float32)

            feature_df["candidate_brand_seen_in_prior"] = np.fromiter((bool(brand) and brand_count_map.get(qid, {}).get(brand, 0) > 0 for qid, brand in zip(query_ids, brands)), dtype=np.int8, count=len(feature_df))
            feature_df["candidate_brand_prior_interaction_count"] = np.fromiter((brand_count_map.get(qid, {}).get(brand, 0) if brand else 0 for qid, brand in zip(query_ids, brands)), dtype=np.float32, count=len(feature_df))
            feature_df["candidate_brand_prior_interaction_count_log1p"] = np.log1p(
                feature_df["candidate_brand_prior_interaction_count"].clip(lower=0.0)
            ).astype(np.float32)
            feature_df["candidate_brand_prior_unique_item_count"] = np.fromiter((brand_unique_item_map.get(qid, {}).get(brand, 0) if brand else 0 for qid, brand in zip(query_ids, brands)), dtype=np.float32, count=len(feature_df))
            history_event_map = query_profiles_df.set_index("query_id")["profile_history_event_n"].astype(float).to_dict()
            feature_df["candidate_brand_prior_share"] = np.fromiter((
                float(brand_count_map.get(qid, {}).get(brand, 0) / max(history_event_map.get(qid, 0.0), 1.0)) if brand else 0.0
                for qid, brand in zip(query_ids, brands)
            ), dtype=np.float32, count=len(feature_df))
            feature_df["candidate_brand_recency_weight"] = np.fromiter((
                math.exp(-max((target_ts - brand_last_map.get(qid, {}).get(brand, 0)) / 86_400_000.0, 0.0) / RECENCY_HALF_LIFE_DAYS)
                if brand and brand in brand_last_map.get(qid, {}) else 0.0
                for qid, brand, target_ts in zip(query_ids, brands, target_times)
            ), dtype=np.float32, count=len(feature_df))
            feature_df["candidate_brand_is_dominant_prior_brand"] = np.fromiter((bool(brand) and brand in dominant_brand_map.get(qid, set()) for qid, brand in zip(query_ids, brands)), dtype=np.int8, count=len(feature_df))
            feature_df["user_prior_unique_brand_count"] = feature_df["query_id"].map(profile_index["user_prior_unique_brand_count"]).fillna(0).astype(np.float32)
            feature_df["user_prior_brand_entropy_norm"] = feature_df["query_id"].map(profile_index["user_prior_brand_entropy_norm"]).fillna(0.0).astype(np.float32)
            feature_df["candidate_brand_share_x_brand_concentration"] = (
                feature_df["candidate_brand_prior_share"]
                * (1.0 - feature_df["user_prior_brand_entropy_norm"].clip(0.0, 1.0))
            ).astype(np.float32)

            PRIOR_FEATURE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
            feature_df.to_parquet(PRIOR_FEATURE_CACHE_PATH, index=False)
            write_json(PRIOR_FEATURE_CACHE_MANIFEST_PATH, expected_prior_feature_contract)
else:
    query_profiles_df = query_meta_df[["query_id", "regime"]].copy()
    for column in ["profile_history_event_n", "profile_history_item_n", "user_prior_unique_brand_count"]:
        query_profiles_df[column] = 0
    for column in ["prior_event_count_log1p", "prior_unique_item_count_log1p", "history_item_count_log1p", "history_strength", "profile_entropy_mean", "profile_concentration", "user_prior_brand_entropy_norm"]:
        query_profiles_df[column] = 0.0


In [12]:
# ==== GAM Feature Contract and Fail-fast Parity Checks ====
if CONDITION_KEY == "s2q":
    SPLINE_FEATURES = list(BASE_SPLINE_FEATURES)
    LINEAR_FEATURES = list(BASE_LINEAR_FEATURES)
    MODEL_FEATURES = list(S2Q_FEATURES)
else:
    SPLINE_FEATURES = list(BASE_SPLINE_FEATURES) + list(PRIOR_SPLINE_FEATURES)
    LINEAR_FEATURES = list(BASE_LINEAR_FEATURES) + list(PRIOR_LINEAR_FEATURES)
    MODEL_FEATURES = list(SHARED_ALL_PRIOR_FEATURES)
if MODEL_FEATURES != list(dict.fromkeys(MODEL_FEATURES)):
    raise RuntimeError("Duplicate model feature columns are not allowed.")
missing_features = [feature for feature in MODEL_FEATURES if feature not in feature_df.columns]
if missing_features:
    raise RuntimeError(f"Missing GAM model features: {missing_features}")
for feature in MODEL_FEATURES:
    feature_df[feature] = pd.to_numeric(feature_df[feature], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)

# Previously-reviewed-item features remain available only for blocking QC and
# labelled sensitivity analysis; they are prohibited from the primary registry.
leaked_exact_item_features = sorted(set(MODEL_FEATURES) & set(EXACT_ITEM_DIAGNOSTIC_FEATURES))
if leaked_exact_item_features:
    raise RuntimeError(
        f"Previously-reviewed-item features entered the primary GAM model: {leaked_exact_item_features}"
    )

positive_exact_seen_n = 0
positive_exact_recency_n = 0
if USE_USER_PRIOR_FEATURES:
    missing_exact_diagnostics = [
        feature for feature in EXACT_ITEM_DIAGNOSTIC_FEATURES
        if feature not in feature_df.columns
    ]
    if missing_exact_diagnostics:
        raise RuntimeError(
            f"Missing previously-reviewed-item diagnostic columns: {missing_exact_diagnostics}"
        )
    for feature in EXACT_ITEM_DIAGNOSTIC_FEATURES:
        values = pd.to_numeric(feature_df[feature], errors="coerce")
        if values.isna().any() or not np.isfinite(values.to_numpy(dtype=float)).all():
            raise RuntimeError(f"Non-finite previously-reviewed-item diagnostic values: {feature}")
        feature_df[feature] = values.astype(np.float32)
    positive_rows = feature_df["is_gt"].eq(1)
    positive_exact_seen_n = int(
        feature_df.loc[positive_rows, "user_item_seen"].ne(0).sum()
    )
    positive_exact_recency_n = int(
        feature_df.loc[positive_rows, "user_item_recency_strength"].ne(0).sum()
    )
    if positive_exact_seen_n != 0 or positive_exact_recency_n != 0:
        raise RuntimeError(
            "The next-novel-item positive target has non-zero previously-reviewed-item diagnostics."
        )

feature_dtypes = {feature: str(feature_df[feature].dtype) for feature in MODEL_FEATURES}
forbidden_model_columns = {
    "is_gt", "target_item_id", "target_timestamp_ms", "query_text", "candidate_item_id",
    "raw_prior_review_text", "prior_review_text", "target_review_text", "rating", "sentiment", "helpful_vote",
}
raw_review_columns_in_model_input = sorted(set(MODEL_FEATURES) & forbidden_model_columns)
if raw_review_columns_in_model_input:
    raise RuntimeError(f"Forbidden review, target, or identifier features: {raw_review_columns_in_model_input}")
query_constant_model_features = sorted(set(MODEL_FEATURES) & set(QUERY_CONSTANT_DIAGNOSTIC_FEATURES))
if query_constant_model_features:
    raise RuntimeError(
        "Query-constant diagnostics cannot directly change within-query order in an additive GAM: "
        f"{query_constant_model_features}"
    )
user_brand_feature_columns = [feature for feature in MODEL_FEATURES if feature in BRAND_HISTORY_FEATURES]
user_brand_feature_columns_S2Q = user_brand_feature_columns if CONDITION_KEY == "s2q" else []
if user_brand_feature_columns_S2Q != []:
    raise RuntimeError("user_brand_feature_columns_S2Q must be empty.")
if CONDITION_KEY != "s2q" and user_brand_feature_columns != BRAND_HISTORY_FEATURES:
    raise RuntimeError("All-Prior brand feature contract is incomplete or out of order.")

candidate_identity_after_feature_hash = stable_frame_hash(
    feature_df, ["query_id", "candidate_item_id", "candidate_rank"], ["query_id", "candidate_rank", "candidate_item_id"]
)
if candidate_identity_after_feature_hash != candidate_identity_before_hash or len(feature_df) != len(candidate_df):
    raise RuntimeError("Candidate rows or IDs changed during feature construction.")

FEATURE_CONTRACT = {
    "contract_version": GAM_FEATURE_CONTRACT_VERSION,
    "category_id": CATEGORY_ID,
    "condition": CONDITION_KEY,
    "candidate_pool_role": CANDIDATE_POOL_ROLE,
    "all_prior_enabled": USE_USER_PRIOR_FEATURES,
    "all_prior_definition": ALL_PRIOR_DEFINITION if USE_USER_PRIOR_FEATURES else "not_loaded",
    "specification_role": SPECIFICATION_ROLE,
    "primary_registry_policy_version": PRIMARY_REGISTRY_POLICY_VERSION,
    "exact_item_familiarity_enabled": ENABLE_EXACT_ITEM_FAMILIARITY,
    "model_features": MODEL_FEATURES,
    "feature_dtypes": feature_dtypes,
    "spline_features": SPLINE_FEATURES,
    "linear_features": LINEAR_FEATURES,
    "missing_value_policy": "numeric_nonfinite_to_zero_float32",
    "n_folds": GAM_N_FOLDS,
    "n_knots": GAM_N_KNOTS,
    "spline_degree": GAM_SPLINE_DEGREE,
    "fixed_c": GAM_C,
    "max_iter": GAM_MAX_ITER,
    "tol": GAM_TOL,
    "class_weight": GAM_CLASS_WEIGHT,
    "solver": GAM_SOLVER,
    "matched_fitting_policy": "target_present_non_cold_queries_only",
    "training_negative_policy": TRAINING_NEGATIVE_POLICY,
    "random_seed": RANDOM_SEED,
}
FEATURE_CONTRACT_HASH = stable_hash_json({
    key: value for key, value in FEATURE_CONTRACT.items()
    if key not in {"category_id", "condition", "candidate_pool_role"}
})

feature_schema_equality_passed = True
if CONDITION_KEY == "s2p":
    SHARED_ALL_PRIOR_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    shared_schema = {
        "contract_version": GAM_FEATURE_CONTRACT_VERSION,
        "category_id": CATEGORY_ID,
        "feature_columns_S2P": MODEL_FEATURES,
        "feature_dtypes_S2P": feature_dtypes,
        "feature_contract_hash": FEATURE_CONTRACT_HASH,
        "specification_role": SPECIFICATION_ROLE,
        "primary_registry_policy_version": PRIMARY_REGISTRY_POLICY_VERSION,
        "exact_item_familiarity_enabled": ENABLE_EXACT_ITEM_FAMILIARITY,
        "missing_value_policy": "numeric_nonfinite_to_zero_float32",
    }
    feature_columns_S2P = list(MODEL_FEATURES)
    feature_dtypes_S2P = dict(feature_dtypes)
    write_json(SHARED_ALL_PRIOR_SCHEMA_PATH, shared_schema)
elif CONDITION_KEY == "full":
    if not SHARED_ALL_PRIOR_SCHEMA_PATH.exists():
        raise FileNotFoundError(f"Run RankP first; missing shared feature schema: {SHARED_ALL_PRIOR_SCHEMA_PATH}")
    shared_schema = load_json(SHARED_ALL_PRIOR_SCHEMA_PATH)
    feature_columns_S2P = list(shared_schema.get("feature_columns_S2P", []))
    feature_columns_Full = list(MODEL_FEATURES)
    feature_dtypes_S2P = dict(shared_schema.get("feature_dtypes_S2P", {}))
    feature_dtypes_Full = dict(feature_dtypes)
    feature_schema_equality_passed = bool(
        shared_schema.get("category_id") == CATEGORY_ID
        and feature_columns_S2P == feature_columns_Full
        and feature_dtypes_S2P == feature_dtypes_Full
        and shared_schema.get("feature_contract_hash") == FEATURE_CONTRACT_HASH
        and shared_schema.get("specification_role") == SPECIFICATION_ROLE
        and shared_schema.get("primary_registry_policy_version") == PRIMARY_REGISTRY_POLICY_VERSION
        and shared_schema.get("exact_item_familiarity_enabled") is False
    )
    if not feature_schema_equality_passed:
        raise RuntimeError("feature_columns_S2P == feature_columns_Full or feature_dtypes parity failed.")

used_model_features_df = pd.DataFrame([
    {
        "feature": feature,
        "transform": "spline" if feature in SPLINE_FEATURES else "linear",
        "feature_role": "brand_history" if feature in BRAND_HISTORY_FEATURES else ("candidate_brand" if feature == "candidate_brand_present" else ("user_prior" if feature in PRIOR_FEATURES else "query_item_relevance")),
        "dtype": feature_dtypes[feature],
    }
    for feature in MODEL_FEATURES
])
users_with_prior_brand = int(query_profiles_df.get("user_prior_unique_brand_count", pd.Series(0, index=query_profiles_df.index)).gt(0).sum())
candidate_brand_prior_match_rate = float(feature_df.get("candidate_brand_seen_in_prior", pd.Series(0, index=feature_df.index)).mean()) if USE_USER_PRIOR_FEATURES else 0.0
contract_diagnostics_df = pd.DataFrame([
    {"check": "candidate_rows", "value": int(len(feature_df)), "passed": True},
    {"check": "cases", "value": int(query_meta_df["case_id"].nunique()), "passed": True},
    {"check": "candidate_brand_non_null_rate", "value": candidate_brand_nonnull_rate, "passed": candidate_brand_nonnull_rate >= 0.0},
    {"check": "users_with_at_least_one_prior_brand", "value": users_with_prior_brand, "passed": True},
    {"check": "candidate_brand_prior_match_rate", "value": candidate_brand_prior_match_rate, "passed": True},
    {"check": "cold_user_count", "value": int(query_meta_df["regime"].eq("cold").sum()), "passed": True},
    {"check": "s2q_feature_count", "value": len(S2Q_FEATURES), "passed": len(S2Q_FEATURES) == EXPECTED_S2Q_FEATURE_COUNT},
    {"check": "s2p_feature_count", "value": len(SHARED_ALL_PRIOR_FEATURES), "passed": len(SHARED_ALL_PRIOR_FEATURES) == EXPECTED_ALL_PRIOR_FEATURE_COUNT},
    {"check": "full_feature_count", "value": len(SHARED_ALL_PRIOR_FEATURES), "passed": len(SHARED_ALL_PRIOR_FEATURES) == EXPECTED_ALL_PRIOR_FEATURE_COUNT},
    {"check": "brand_aware_feature_count", "value": int(1 + (len(BRAND_HISTORY_FEATURES) if USE_USER_PRIOR_FEATURES else 0)), "passed": True},
    {"check": "positive_target_user_item_seen_nonzero", "value": positive_exact_seen_n, "passed": positive_exact_seen_n == 0},
    {"check": "positive_target_user_item_recency_nonzero", "value": positive_exact_recency_n, "passed": positive_exact_recency_n == 0},
    {"check": "previously_reviewed_item_features_in_primary_model", "value": len(leaked_exact_item_features), "passed": len(leaked_exact_item_features) == 0},
    {"check": "s2p_full_feature_schema_equality", "value": feature_schema_equality_passed, "passed": feature_schema_equality_passed},
    {"check": "temporal_validation", "value": temporal_validation_passed, "passed": temporal_validation_passed},
    {"check": "same_target_item_prior_rows", "value": same_target_item_prior_rows, "passed": same_target_item_prior_rows == 0},
    {"check": "raw_review_columns_in_model_input", "value": len(raw_review_columns_in_model_input), "passed": len(raw_review_columns_in_model_input) == 0},
    {"check": "query_constant_model_features", "value": len(query_constant_model_features), "passed": len(query_constant_model_features) == 0},
    {"check": "brand_terms_added_to_synthetic_query", "value": brand_terms_added_to_synthetic_query_n, "passed": brand_terms_added_to_synthetic_query_n == 0},
    {"check": "candidate_identity_unchanged", "value": candidate_identity_after_feature_hash == candidate_identity_before_hash, "passed": candidate_identity_after_feature_hash == candidate_identity_before_hash},
])
if not contract_diagnostics_df["passed"].all():
    raise RuntimeError(f"Feature contract diagnostics failed: {contract_diagnostics_df.loc[~contract_diagnostics_df['passed'], 'check'].tolist()}")
print("Model features:", MODEL_FEATURES)
print("Feature contract hash:", FEATURE_CONTRACT_HASH)


Model features: ['candidate_rank_pct', 'candidate_score_norm', 'query_item_structured_match', 'item_prequery_review_count_log1p', 'item_last_review_gap_days', 'candidate_brand_present']
Feature contract hash: 34c37b221959366e44fcbb03af63dc3ec51976744ffd809548086d8176033ba8


## 5. Fit the Spline GAM on the Max-Depth Interface

In [13]:
# ==== GAM-lite Model ====
class AdditiveSplineDesign:
    def __init__(self, spline_features, linear_features, n_knots, degree):
        self.spline_features = list(spline_features)
        self.linear_features = list(linear_features)
        self.n_knots = int(n_knots)
        self.degree = int(degree)
        self.transformers = {}
        self.feature_slices = {}
        self.active_spline_features = []
        self.active_linear_features = []
        self.n_output_features_ = 0

    def fit_transform(self, frame: pd.DataFrame):
        matrices = []
        start = 0
        for feature in self.spline_features:
            values = frame[[feature]].to_numpy(dtype=np.float64)
            if np.nanstd(values) <= 1e-10:
                continue
            transformer = SplineTransformer(
                n_knots=self.n_knots,
                degree=self.degree,
                knots="quantile",
                extrapolation="constant",
                include_bias=False,
                sparse_output=True,
            )
            matrix = transformer.fit_transform(values)
            matrix = sparse.csr_matrix(matrix)
            stop = start + matrix.shape[1]
            self.transformers[feature] = transformer
            self.feature_slices[feature] = slice(start, stop)
            self.active_spline_features.append(feature)
            matrices.append(matrix)
            start = stop

        for feature in self.linear_features:
            values = frame[[feature]].to_numpy(dtype=np.float64)
            if np.nanstd(values) <= 1e-10:
                continue
            matrix = sparse.csr_matrix(values)
            stop = start + 1
            self.feature_slices[feature] = slice(start, stop)
            self.active_linear_features.append(feature)
            matrices.append(matrix)
            start = stop

        self.n_output_features_ = start
        if not matrices:
            return sparse.csr_matrix((len(frame), 0), dtype=np.float64)
        return sparse.hstack(matrices, format="csr")

    def transform(self, frame: pd.DataFrame):
        matrices = []
        for feature in self.active_spline_features:
            matrices.append(sparse.csr_matrix(self.transformers[feature].transform(frame[[feature]].to_numpy(dtype=np.float64))))
        for feature in self.active_linear_features:
            matrices.append(sparse.csr_matrix(frame[[feature]].to_numpy(dtype=np.float64)))
        if not matrices:
            return sparse.csr_matrix((len(frame), 0), dtype=np.float64)
        return sparse.hstack(matrices, format="csr")


def deterministic_query_sample(group: pd.DataFrame, seed: int) -> tuple[np.ndarray, np.ndarray]:
    positive = group.index[group["is_gt"].eq(1)].to_numpy()
    if len(positive) != 1:
        return np.asarray([], dtype=np.int64), np.asarray([], dtype=np.float64)

    negatives = group.loc[group["is_gt"].eq(0)].sort_values(["candidate_rank", "candidate_item_id"], kind="mergesort")
    hard = negatives.head(GAM_HARD_NEGATIVES_PER_QUERY).index.to_numpy()
    remaining = negatives.loc[~negatives.index.isin(hard)]

    rng = np.random.default_rng(seed)
    stratified = []
    if len(remaining):
        bins = np.array_split(remaining.index.to_numpy(), GAM_NEGATIVE_BINS)
        per_bin = max(1, math.ceil(GAM_STRATIFIED_NEGATIVES_PER_QUERY / max(len(bins), 1)))
        for bin_indices in bins:
            if len(bin_indices) == 0:
                continue
            n_take = min(per_bin, len(bin_indices))
            stratified.extend(rng.choice(bin_indices, size=n_take, replace=False).tolist())
    stratified = np.asarray(stratified[:GAM_STRATIFIED_NEGATIVES_PER_QUERY], dtype=np.int64)

    selected_negatives = np.asarray(list(dict.fromkeys([*hard.tolist(), *stratified.tolist()])), dtype=np.int64)
    selected = np.concatenate([positive.astype(np.int64), selected_negatives])
    weights = np.zeros(len(selected), dtype=np.float64)
    weights[0] = 0.5
    if len(selected_negatives):
        weights[1:] = 0.5 / len(selected_negatives)
    return selected, weights


def build_training_sample(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    sample_indices = []
    sample_weights = []
    qc_rows = []
    for query_id, group in tqdm(frame.groupby("query_id", sort=False), total=frame["query_id"].nunique(), desc="Select GAM training rows"):
        query_seed = RANDOM_SEED + int(stable_hash_text(query_id)[:8], 16)
        selected, weights = deterministic_query_sample(group, query_seed)
        sample_indices.extend(selected.tolist())
        sample_weights.extend(weights.tolist())
        qc_rows.append({
            "query_id": query_id,
            "gt_in_pool": int(group["is_gt"].max()),
            "sampled_rows": int(len(selected)),
            "sampled_negatives": int(max(len(selected) - 1, 0)),
        })
    sampled = frame.loc[sample_indices].copy()
    sampled["gam_sample_weight"] = np.asarray(sample_weights, dtype=np.float64)
    return sampled, pd.DataFrame(qc_rows)


def predict_in_chunks(model, design, frame: pd.DataFrame, chunk_size: int) -> np.ndarray:
    predictions = np.empty(len(frame), dtype=np.float32)
    for start in range(0, len(frame), chunk_size):
        stop = min(start + chunk_size, len(frame))
        matrix = design.transform(frame.iloc[start:stop])
        predictions[start:stop] = model.predict_proba(matrix)[:, 1].astype(np.float32)
    return predictions


def build_effect_grids(frame: pd.DataFrame, features: list[str], points: int) -> dict[str, np.ndarray]:
    grids = {}
    for feature in features:
        values = pd.to_numeric(frame[feature], errors="coerce").dropna().to_numpy(dtype=float)
        if values.size == 0:
            grids[feature] = np.asarray([0.0], dtype=float)
            continue
        low, high = np.quantile(values, [0.01, 0.99])
        if not np.isfinite(low) or not np.isfinite(high) or abs(high - low) <= 1e-12:
            grids[feature] = np.asarray([float(np.nanmedian(values))], dtype=float)
        else:
            grids[feature] = np.linspace(float(low), float(high), int(points))
    return grids


def extract_effect_rows(model, design: AdditiveSplineDesign, effect_grids: dict[str, np.ndarray], fold: int) -> list[dict]:
    coef = np.asarray(model.coef_).ravel()
    rows = []
    for feature, grid in effect_grids.items():
        if feature not in design.feature_slices:
            continue
        feature_slice = design.feature_slices[feature]
        feature_coef = coef[feature_slice]
        if feature in design.transformers:
            basis = sparse.csr_matrix(design.transformers[feature].transform(grid.reshape(-1, 1)))
            effect = np.asarray(basis @ feature_coef).ravel()
        else:
            effect = grid * float(feature_coef[0])
        reference = float(np.interp(np.median(grid), grid, effect)) if len(grid) > 1 else float(effect[0])
        centered = effect - reference
        for value, effect_value in zip(grid, centered):
            rows.append({
                "fold": int(fold),
                "feature": feature,
                "grid_value": float(value),
                "centered_additive_effect": float(effect_value),
            })
    return rows

In [14]:
# ==== Matched OOF Training for Base ====
MATCHED_FITTING_POLICY = "target_present_non_cold_queries_only"
MODEL_PARAMETER_CONTRACT = {
    "n_folds": GAM_N_FOLDS,
    "n_knots": GAM_N_KNOTS,
    "spline_degree": GAM_SPLINE_DEGREE,
    "fixed_c": GAM_C,
    "solver": GAM_SOLVER,
    "penalty": "l2",
    "max_iter": GAM_MAX_ITER,
    "tol": GAM_TOL,
    "class_weight": GAM_CLASS_WEIGHT,
    "training_negative_policy": TRAINING_NEGATIVE_POLICY,
    "hard_negatives_per_query": GAM_HARD_NEGATIVES_PER_QUERY,
    "stratified_negatives_per_query": GAM_STRATIFIED_NEGATIVES_PER_QUERY,
    "negative_bins": GAM_NEGATIVE_BINS,
    "fitting_policy": MATCHED_FITTING_POLICY,
    "fit_pool_depth": FIT_POOL_DEPTH,
    "random_seed": RANDOM_SEED,
}
MODEL_PARAMETER_HASH = stable_hash_json(MODEL_PARAMETER_CONTRACT)


def canonical_query_frame() -> pd.DataFrame:
    frame = feature_df[
        ["query_id", "user_id", "regime", "target_timestamp_ms", "gt_in_pool"]
    ].drop_duplicates("query_id").copy()
    frame["query_id"] = frame["query_id"].astype(str)
    frame["user_id"] = frame["user_id"].fillna("").astype(str)
    frame["regime"] = frame["regime"].map(normalize_regime)
    if frame["user_id"].eq("").any() or frame["user_id"].nunique() < GAM_N_FOLDS:
        raise RuntimeError("Insufficient non-empty user groups for five-fold GAM OOF.")
    return frame.sort_values(["query_id"], kind="mergesort").reset_index(drop=True)


def build_query_fold_map(query_frame: pd.DataFrame) -> pd.DataFrame:
    splitter = GroupKFold(n_splits=GAM_N_FOLDS)
    rows = []
    for fold, (_, test_idx) in enumerate(
        splitter.split(query_frame, groups=query_frame["user_id"].astype(str)), start=1
    ):
        rows.extend(
            {
                "query_id": str(row.query_id),
                "user_id": str(row.user_id),
                "regime": str(row.regime),
                "cv_fold": int(fold),
            }
            for row in query_frame.iloc[test_idx].itertuples(index=False)
        )
    result = pd.DataFrame(rows).sort_values("query_id", kind="mergesort").reset_index(drop=True)
    if result["query_id"].nunique() != len(query_frame):
        raise RuntimeError("Canonical fold map does not cover each query exactly once.")
    if result.groupby("user_id")["cv_fold"].nunique().max() != 1:
        raise RuntimeError("Canonical GAM folds are not user-group disjoint.")
    return result


query_split_df = canonical_query_frame()
query_universe_contract_hash = stable_frame_hash(
    query_split_df,
    ["query_id", "user_id", "regime"],
    ["query_id"],
)

if CONDITION_KEY == "full":
    if not SHARED_ALL_PRIOR_CONTRACT_PATH.exists() or not SHARED_ALL_PRIOR_FOLD_ASSIGNMENTS_PATH.exists():
        raise FileNotFoundError("Run the matching category 12b notebook first; shared RankP GAM artifacts are missing.")
    shared_model_contract = load_json(SHARED_ALL_PRIOR_CONTRACT_PATH)
    required_contract = {
        "contract_version": "matched_gam_models_v3_primary6_15",
        "category_id": CATEGORY_ID,
        "specification_role": SPECIFICATION_ROLE,
        "primary_registry_policy_version": PRIMARY_REGISTRY_POLICY_VERSION,
        "exact_item_familiarity_enabled": ENABLE_EXACT_ITEM_FAMILIARITY,
        "feature_contract_hash": FEATURE_CONTRACT_HASH,
        "feature_columns": MODEL_FEATURES,
        "feature_dtypes": feature_dtypes,
        "query_set_hash": query_set_hash,
        "query_universe_contract_hash": query_universe_contract_hash,
        "fit_pool_depth": FIT_POOL_DEPTH,
        "n_folds": GAM_N_FOLDS,
        "model_parameter_hash": MODEL_PARAMETER_HASH,
        "matched_fitting_policy": MATCHED_FITTING_POLICY,
    }
    mismatches = {
        key: (shared_model_contract.get(key), expected)
        for key, expected in required_contract.items()
        if shared_model_contract.get(key) != expected
    }
    if mismatches:
        raise RuntimeError(f"Full does not match the exact RankP model contract: {mismatches}")

    query_fold_df = pd.read_parquet(SHARED_ALL_PRIOR_FOLD_ASSIGNMENTS_PATH)
    require_columns(query_fold_df, ["query_id", "cv_fold"], "S2-P fold assignments")
    query_fold_df["query_id"] = query_fold_df["query_id"].astype(str)
    if set(query_fold_df["query_id"]) != set(query_split_df["query_id"]):
        raise RuntimeError("RankP and Full query universes differ.")
    query_fold_hash = stable_frame_hash(query_fold_df, ["query_id", "cv_fold"], ["query_id"])
    if query_fold_hash != shared_model_contract.get("query_fold_hash"):
        raise RuntimeError("RankP fold-assignment hash does not match its contract.")

    feature_df["cv_fold"] = feature_df["query_id"].astype(str).map(
        query_fold_df.set_index("query_id")["cv_fold"]
    ).astype(int)
    feature_df["gam_score"] = np.nan
    feature_df["prediction_source"] = "exact_shared_s2p_gam"
    effect_grids = build_effect_grids(feature_df, MODEL_FEATURES, GAM_EFFECT_GRID_POINTS)
    effect_rows = []
    cv_rows = list(shared_model_contract.get("cv_rows", []))
    model_file_records = list(shared_model_contract.get("model_files", []))
    if len(model_file_records) != GAM_N_FOLDS:
        raise RuntimeError("RankP contract does not list exactly five fitted fold models.")

    with runtime_step("gam_shared_s2p_model_loading_and_scoring"):
        for fold_record in sorted(model_file_records, key=lambda row: int(row["fold"])):
            fold = int(fold_record["fold"])
            model_path = SHARED_ALL_PRIOR_MODEL_DIR / str(fold_record["file_name"])
            if not model_path.exists():
                raise FileNotFoundError(f"Missing RankP fold model: {model_path}")
            if file_sha256(model_path) != fold_record["sha256"]:
                raise RuntimeError(f"RankP fold model SHA256 mismatch for fold {fold}.")
            with model_path.open("rb") as handle:
                payload = pickle.load(handle)
            payload_expected = {
                "category_id": CATEGORY_ID,
                "fold": fold,
                "feature_contract_hash": FEATURE_CONTRACT_HASH,
                "model_parameter_hash": MODEL_PARAMETER_HASH,
                "matched_training_membership_hash": shared_model_contract["matched_training_membership_hash"],
                "query_fold_hash": query_fold_hash,
            }
            payload_mismatches = {
                key: (payload.get(key), value)
                for key, value in payload_expected.items()
                if payload.get(key) != value
            }
            if payload_mismatches:
                raise RuntimeError(f"RankP fold payload mismatch for fold {fold}: {payload_mismatches}")
            test_frame = feature_df[feature_df["cv_fold"].eq(fold)].copy()
            feature_df.loc[test_frame.index, "gam_score"] = predict_in_chunks(
                payload["model"], payload["design"], test_frame, GAM_SCORE_CHUNK_SIZE
            )
            effect_rows.extend(
                extract_effect_rows(payload["model"], payload["design"], effect_grids, fold)
            )

    if feature_df["gam_score"].isna().any():
        raise RuntimeError("Full scoring with exact RankP fold models produced missing scores.")
    model_cv_summary_df = pd.DataFrame(cv_rows)
    gam_effect_curves_df = pd.DataFrame(effect_rows)
    negative_sampling_qc_df = pd.DataFrame([{
        "policy": "not_retrained_load_exact_s2p_models",
        "n_rows": 0,
        "n_queries": 0,
    }])
    shared_training_rows_hash = shared_model_contract["training_rows_hash"]
    shared_labels_hash = shared_model_contract["labels_hash"]
    matched_training_membership_hash = shared_model_contract["matched_training_membership_hash"]
    fit_query_membership_hash = shared_model_contract["fit_query_membership_hash"]
    sample_weight_hash = shared_model_contract["sample_weight_hash"]
    training_rows_used = 0
    n_cold_training_queries = 0

else:
    # 12a is the single writer of the category-wide fold map and the GAM-specific
    # sampled fitting membership. 12b is a strict reader. This keeps the a/b
    # ablation matched without imposing GAM negative sampling on other rerankers.
    if CONDITION_KEY == "s2q":
        query_fold_df = build_query_fold_map(query_split_df)
        COMMON_RERANK_CONTRACT_DIR.mkdir(parents=True, exist_ok=True)
        query_fold_df.to_parquet(COMMON_QUERY_FOLD_ASSIGNMENTS_PATH, index=False)
        query_fold_hash = stable_frame_hash(query_fold_df, ["query_id", "cv_fold"], ["query_id"])
        common_fold_manifest = {
            "contract_version": "category_common_user_group_folds_v1",
            "category_id": CATEGORY_ID,
            "query_set_hash": query_set_hash,
            "query_universe_contract_hash": query_universe_contract_hash,
            "n_folds": GAM_N_FOLDS,
            "query_fold_hash": query_fold_hash,
            "user_group_disjoint": True,
            "complete_evaluation_universe": True,
        }
        write_json(COMMON_QUERY_FOLD_MANIFEST_PATH, common_fold_manifest)
    else:
        if not COMMON_QUERY_FOLD_ASSIGNMENTS_PATH.exists() or not COMMON_QUERY_FOLD_MANIFEST_PATH.exists():
            raise FileNotFoundError("Run the matching category 12a notebook first; common five-fold map is missing.")
        common_fold_manifest = load_json(COMMON_QUERY_FOLD_MANIFEST_PATH)
        expected_fold_contract = {
            "category_id": CATEGORY_ID,
            "query_set_hash": query_set_hash,
            "query_universe_contract_hash": query_universe_contract_hash,
            "n_folds": GAM_N_FOLDS,
            "user_group_disjoint": True,
            "complete_evaluation_universe": True,
        }
        fold_mismatches = {
            key: (common_fold_manifest.get(key), value)
            for key, value in expected_fold_contract.items()
            if common_fold_manifest.get(key) != value
        }
        if fold_mismatches:
            raise RuntimeError(f"12a/12b common fold contract mismatch: {fold_mismatches}")
        query_fold_df = pd.read_parquet(COMMON_QUERY_FOLD_ASSIGNMENTS_PATH)
        require_columns(query_fold_df, ["query_id", "user_id", "regime", "cv_fold"], "Common fold map")
        query_fold_df["query_id"] = query_fold_df["query_id"].astype(str)
        if set(query_fold_df["query_id"]) != set(query_split_df["query_id"]):
            raise RuntimeError("12a/12b query universes differ.")
        query_fold_hash = stable_frame_hash(query_fold_df, ["query_id", "cv_fold"], ["query_id"])
        if query_fold_hash != common_fold_manifest.get("query_fold_hash"):
            raise RuntimeError("Common fold-map bytes do not reproduce the manifest hash.")

    feature_df["cv_fold"] = feature_df["query_id"].astype(str).map(
        query_fold_df.set_index("query_id")["cv_fold"]
    ).astype(int)

    if CONDITION_KEY == "s2q":
        eligible_training_rows = feature_df[
            feature_df["gt_in_pool"].eq(1) & ~feature_df["regime"].eq("cold")
        ].copy()
        training_sample_df, negative_sampling_qc_df = build_training_sample(eligible_training_rows)
        training_sample_df["sample_role"] = np.where(
            training_sample_df["is_gt"].eq(1), "positive", "deterministic_hard_or_stratified_negative"
        )
        training_membership_df = training_sample_df[
            ["query_id", "candidate_item_id", "is_gt", "cv_fold", "gam_sample_weight", "sample_role"]
        ].copy()
        training_membership_df = training_membership_df.sort_values(
            ["query_id", "candidate_item_id"], kind="mergesort"
        ).reset_index(drop=True)
        COMMON_RERANK_CONTRACT_DIR.mkdir(parents=True, exist_ok=True)
        training_membership_df.to_parquet(COMMON_GAM_TRAINING_MEMBERSHIP_PATH, index=False)
        matched_training_membership_hash = stable_frame_hash(
            training_membership_df,
            ["query_id", "candidate_item_id", "is_gt", "cv_fold", "gam_sample_weight", "sample_role"],
            ["query_id", "candidate_item_id"],
        )
        fit_query_membership_hash = stable_hash_text(
            "\n".join(sorted(training_membership_df["query_id"].astype(str).unique()))
        )
        sample_weight_hash = stable_frame_hash(
            training_membership_df,
            ["query_id", "candidate_item_id", "gam_sample_weight"],
            ["query_id", "candidate_item_id"],
        )
        base_feature_training_values_hash = stable_frame_hash(
            training_sample_df,
            ["query_id", "candidate_item_id", *S2Q_FEATURES],
            ["query_id", "candidate_item_id"],
        )
        training_membership_manifest = {
            "contract_version": "matched_gam_sampling_v2_primary6",
            "category_id": CATEGORY_ID,
            "candidate_source_signature": candidate_source_signature,
            "query_set_hash": query_set_hash,
            "query_fold_hash": query_fold_hash,
            "fit_pool_depth": FIT_POOL_DEPTH,
            "fitting_policy": MATCHED_FITTING_POLICY,
            "training_negative_policy": TRAINING_NEGATIVE_POLICY,
            "hard_negatives_per_query": GAM_HARD_NEGATIVES_PER_QUERY,
            "stratified_negatives_per_query": GAM_STRATIFIED_NEGATIVES_PER_QUERY,
            "negative_bins": GAM_NEGATIVE_BINS,
            "matched_training_membership_hash": matched_training_membership_hash,
            "fit_query_membership_hash": fit_query_membership_hash,
            "sample_weight_hash": sample_weight_hash,
            "base_feature_training_values_hash": base_feature_training_values_hash,
            "s2q_feature_columns": S2Q_FEATURES,
            "primary_registry_policy_version": PRIMARY_REGISTRY_POLICY_VERSION,
            "n_rows": int(len(training_membership_df)),
            "n_queries": int(training_membership_df["query_id"].nunique()),
        }
        write_json(COMMON_GAM_TRAINING_MEMBERSHIP_MANIFEST_PATH, training_membership_manifest)
    else:
        if not COMMON_GAM_TRAINING_MEMBERSHIP_PATH.exists() or not COMMON_GAM_TRAINING_MEMBERSHIP_MANIFEST_PATH.exists():
            raise FileNotFoundError("Run the matching category 12a notebook first; matched GAM sample is missing.")
        training_membership_manifest = load_json(COMMON_GAM_TRAINING_MEMBERSHIP_MANIFEST_PATH)
        expected_membership_contract = {
            "contract_version": "matched_gam_sampling_v2_primary6",
            "category_id": CATEGORY_ID,
            "candidate_source_signature": candidate_source_signature,
            "query_set_hash": query_set_hash,
            "query_fold_hash": query_fold_hash,
            "fit_pool_depth": FIT_POOL_DEPTH,
            "fitting_policy": MATCHED_FITTING_POLICY,
            "training_negative_policy": TRAINING_NEGATIVE_POLICY,
            "hard_negatives_per_query": GAM_HARD_NEGATIVES_PER_QUERY,
            "stratified_negatives_per_query": GAM_STRATIFIED_NEGATIVES_PER_QUERY,
            "negative_bins": GAM_NEGATIVE_BINS,
            "s2q_feature_columns": S2Q_FEATURES,
            "primary_registry_policy_version": PRIMARY_REGISTRY_POLICY_VERSION,
        }
        membership_mismatches = {
            key: (training_membership_manifest.get(key), value)
            for key, value in expected_membership_contract.items()
            if training_membership_manifest.get(key) != value
        }
        if membership_mismatches:
            raise RuntimeError(f"12a/12b matched sample contract mismatch: {membership_mismatches}")
        training_membership_df = pd.read_parquet(COMMON_GAM_TRAINING_MEMBERSHIP_PATH)
        require_columns(
            training_membership_df,
            ["query_id", "candidate_item_id", "is_gt", "cv_fold", "gam_sample_weight", "sample_role"],
            "Matched GAM training membership",
        )
        training_membership_df["query_id"] = training_membership_df["query_id"].astype(str)
        training_membership_df["candidate_item_id"] = training_membership_df["candidate_item_id"].astype(str)
        matched_training_membership_hash = stable_frame_hash(
            training_membership_df,
            ["query_id", "candidate_item_id", "is_gt", "cv_fold", "gam_sample_weight", "sample_role"],
            ["query_id", "candidate_item_id"],
        )
        fit_query_membership_hash = stable_hash_text(
            "\n".join(sorted(training_membership_df["query_id"].astype(str).unique()))
        )
        sample_weight_hash = stable_frame_hash(
            training_membership_df,
            ["query_id", "candidate_item_id", "gam_sample_weight"],
            ["query_id", "candidate_item_id"],
        )
        for key, observed in {
            "matched_training_membership_hash": matched_training_membership_hash,
            "fit_query_membership_hash": fit_query_membership_hash,
            "sample_weight_hash": sample_weight_hash,
        }.items():
            if observed != training_membership_manifest.get(key):
                raise RuntimeError(f"Matched GAM training artifact does not reproduce {key}.")
        feature_lookup = feature_df.copy()
        feature_lookup["query_id"] = feature_lookup["query_id"].astype(str)
        feature_lookup["candidate_item_id"] = feature_lookup["candidate_item_id"].astype(str)
        training_sample_df = training_membership_df.merge(
            feature_lookup,
            on=["query_id", "candidate_item_id", "is_gt", "cv_fold"],
            how="left",
            validate="one_to_one",
        )
        if len(training_sample_df) != len(training_membership_df) or training_sample_df[MODEL_FEATURES].isna().any().any():
            raise RuntimeError("12b could not reproduce every 12a sampled candidate row.")
        base_feature_training_values_hash = stable_frame_hash(
            training_sample_df,
            ["query_id", "candidate_item_id", *S2Q_FEATURES],
            ["query_id", "candidate_item_id"],
        )
        if base_feature_training_values_hash != training_membership_manifest.get(
            "base_feature_training_values_hash"
        ):
            raise RuntimeError(
                "S2-P does not reproduce the six S2-Q base-feature values on the matched training rows."
            )
        negative_sampling_qc_df = training_membership_df.groupby("query_id", sort=False).agg(
            gt_in_pool=("is_gt", "max"),
            sampled_rows=("candidate_item_id", "size"),
            sampled_negatives=("is_gt", lambda values: int((pd.Series(values).eq(0)).sum())),
        ).reset_index()

    # Final matched-sample checks apply to both 12a and 12b.
    weight_by_query = training_sample_df.groupby("query_id")["gam_sample_weight"].sum()
    positive_by_query = training_sample_df.groupby("query_id")["is_gt"].sum()
    if not np.allclose(weight_by_query.to_numpy(dtype=float), 1.0, atol=1e-8):
        raise RuntimeError("GAM sample weights do not sum to one within every query.")
    if not positive_by_query.eq(1).all():
        raise RuntimeError("Every GAM fitting query must contain exactly one positive candidate.")
    if training_sample_df["regime"].eq("cold").any():
        raise RuntimeError("Cold queries entered matched GAM fitting.")
    if not training_sample_df["gt_in_pool"].eq(1).all():
        raise RuntimeError("Target-absent queries entered matched GAM fitting.")
    n_cold_training_queries = int(training_sample_df.loc[training_sample_df["regime"].eq("cold"), "query_id"].nunique())
    if n_cold_training_queries != 0:
        raise RuntimeError("prior-aware/no-prior matched training contains cold queries.")

    feature_df["gam_score"] = np.nan
    feature_df["prediction_source"] = "native_s2q_gam" if CONDITION_KEY == "s2q" else "shared_s2p_gam"
    effect_grids = build_effect_grids(feature_df, MODEL_FEATURES, GAM_EFFECT_GRID_POINTS)
    cv_rows = []
    effect_rows = []
    model_file_records = []
    fold_model_dir = S2Q_FOLD_MODEL_DIR if CONDITION_KEY == "s2q" else SHARED_ALL_PRIOR_MODEL_DIR
    fold_model_dir.mkdir(parents=True, exist_ok=True)

    with runtime_step("gam_oof_training_and_scoring"):
        for fold in range(1, GAM_N_FOLDS + 1):
            fit_sample = training_sample_df[~training_sample_df["cv_fold"].eq(fold)].copy()
            test_frame = feature_df[feature_df["cv_fold"].eq(fold)].copy()
            if fit_sample["is_gt"].sum() < 2:
                raise RuntimeError(f"Fold {fold} has insufficient positive training queries.")
            fold_fit_membership_hash = stable_frame_hash(
                fit_sample,
                ["query_id", "candidate_item_id", "is_gt", "gam_sample_weight"],
                ["query_id", "candidate_item_id"],
            )
            model_file_name = (
                f"s2q_gam_fold_{fold}.pkl"
                if CONDITION_KEY == "s2q"
                else f"shared_all_prior_gam_fold_{fold}.pkl"
            )
            model_path = fold_model_dir / model_file_name
            payload_expected = {
                "category_id": CATEGORY_ID,
                "condition": CONDITION_KEY,
                "fold": fold,
                "feature_contract_hash": FEATURE_CONTRACT_HASH,
                "model_parameter_hash": MODEL_PARAMETER_HASH,
                "matched_training_membership_hash": matched_training_membership_hash,
                "fold_fit_membership_hash": fold_fit_membership_hash,
                "query_fold_hash": query_fold_hash,
            }
            payload = None
            reused_checkpoint = False
            if GAM_RESUME_VALID_FOLDS and model_path.exists():
                try:
                    with model_path.open("rb") as handle:
                        candidate_payload = pickle.load(handle)
                    checkpoint_mismatches = {
                        key: (candidate_payload.get(key), value)
                        for key, value in payload_expected.items()
                        if candidate_payload.get(key) != value
                    }
                    if not checkpoint_mismatches and not candidate_payload.get("convergence_warning", True):
                        payload = candidate_payload
                        reused_checkpoint = True
                except Exception:
                    payload = None

            if payload is None:
                design = AdditiveSplineDesign(
                    SPLINE_FEATURES,
                    LINEAR_FEATURES,
                    n_knots=GAM_N_KNOTS,
                    degree=GAM_SPLINE_DEGREE,
                )
                x_train = design.fit_transform(fit_sample)
                if x_train.shape[1] == 0:
                    raise RuntimeError("The GAM design matrix has no variable features.")
                model = LogisticRegression(
                    C=GAM_C,
                    solver=GAM_SOLVER,
                    penalty="l2",
                    class_weight=GAM_CLASS_WEIGHT,
                    max_iter=GAM_MAX_ITER,
                    tol=GAM_TOL,
                    random_state=RANDOM_SEED + fold,
                )
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always", ConvergenceWarning)
                    fit_started = time.perf_counter()
                    model.fit(
                        x_train,
                        fit_sample["is_gt"].astype(np.int8).to_numpy(),
                        sample_weight=fit_sample["gam_sample_weight"].to_numpy(dtype=float),
                    )
                    fit_runtime = float(time.perf_counter() - fit_started)
                convergence_warning = any(issubclass(w.category, ConvergenceWarning) for w in caught)
                n_iter = int(np.max(model.n_iter_))
                if convergence_warning or n_iter >= GAM_MAX_ITER:
                    raise RuntimeError(
                        f"GAM fold {fold} did not converge cleanly: warning={convergence_warning}, n_iter={n_iter}."
                    )
                payload = {
                    **payload_expected,
                    "model": model,
                    "design": design,
                    "fit_runtime_sec": fit_runtime,
                    "n_iter": n_iter,
                    "convergence_warning": False,
                }
                with model_path.open("wb") as handle:
                    pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
                del x_train
            else:
                model = payload["model"]
                design = payload["design"]
                fit_runtime = 0.0
                n_iter = int(payload["n_iter"])
                convergence_warning = bool(payload["convergence_warning"])

            score_started = time.perf_counter()
            predictions = predict_in_chunks(model, design, test_frame, GAM_SCORE_CHUNK_SIZE)
            score_runtime = float(time.perf_counter() - score_started)
            feature_df.loc[test_frame.index, "gam_score"] = predictions
            effect_rows.extend(extract_effect_rows(model, design, effect_grids, fold))
            model_sha256 = file_sha256(model_path)
            model_file_records.append({
                "fold": fold,
                "file_name": model_file_name,
                "sha256": model_sha256,
            })
            cv_rows.append({
                "fold": fold,
                "n_train_queries": int(fit_sample["query_id"].nunique()),
                "n_validation_queries": 0,
                "n_train_rows": int(len(fit_sample)),
                "n_test_queries": int(test_frame["query_id"].nunique()),
                "n_test_rows": int(len(test_frame)),
                "n_cold_training_queries": 0,
                "active_spline_features": json.dumps(design.active_spline_features),
                "active_linear_features": json.dumps(design.active_linear_features),
                "design_columns": int(design.n_output_features_),
                "fit_runtime_sec": fit_runtime,
                "score_runtime_sec": score_runtime,
                "n_iter": n_iter,
                "convergence_warning": convergence_warning,
                "reused_valid_checkpoint": reused_checkpoint,
                "fold_fit_membership_hash": fold_fit_membership_hash,
                "model_sha256": model_sha256,
            })
            del fit_sample, test_frame, predictions, model, design, payload
            gc.collect()

    if feature_df["gam_score"].isna().any():
        raise RuntimeError("OOF GAM predictions are missing.")
    model_cv_summary_df = pd.DataFrame(cv_rows)
    gam_effect_curves_df = pd.DataFrame(effect_rows)
    training_hash_columns = ["query_id", "candidate_item_id", "is_gt", *MODEL_FEATURES]
    shared_training_rows_hash = stable_frame_hash(
        training_sample_df, training_hash_columns, ["query_id", "candidate_item_id"]
    )
    shared_labels_hash = stable_frame_hash(
        training_sample_df,
        ["query_id", "candidate_item_id", "is_gt"],
        ["query_id", "candidate_item_id"],
    )
    training_rows_used = int(len(training_sample_df))

    model_contract = {
        "contract_version": "matched_gam_models_v3_primary6_15",
        "category_id": CATEGORY_ID,
        "specification_role": SPECIFICATION_ROLE,
        "primary_registry_policy_version": PRIMARY_REGISTRY_POLICY_VERSION,
        "exact_item_familiarity_enabled": ENABLE_EXACT_ITEM_FAMILIARITY,
        "source_condition": CONDITION_KEY,
        "candidate_pool_role": "baseline_query_only",
        "feature_contract_hash": FEATURE_CONTRACT_HASH,
        "feature_columns": MODEL_FEATURES,
        "feature_dtypes": feature_dtypes,
        "missing_value_policy": "numeric_nonfinite_to_zero_float32",
        "training_rows_hash": shared_training_rows_hash,
        "labels_hash": shared_labels_hash,
        "matched_training_membership_hash": matched_training_membership_hash,
        "fit_query_membership_hash": fit_query_membership_hash,
        "sample_weight_hash": sample_weight_hash,
        "query_fold_hash": query_fold_hash,
        "query_set_hash": query_set_hash,
        "query_universe_contract_hash": query_universe_contract_hash,
        "matched_fitting_policy": MATCHED_FITTING_POLICY,
        "fit_pool_depth": FIT_POOL_DEPTH,
        "n_folds": GAM_N_FOLDS,
        "model_parameter_hash": MODEL_PARAMETER_HASH,
        "model_parameters": MODEL_PARAMETER_CONTRACT,
        "cv_rows": cv_rows,
        "model_files": model_file_records,
    }
    if CONDITION_KEY == "s2q":
        write_json(S2Q_MODEL_CONTRACT_PATH, model_contract)
    else:
        query_fold_df[["query_id", "cv_fold"]].to_parquet(
            SHARED_ALL_PRIOR_FOLD_ASSIGNMENTS_PATH, index=False
        )
        write_json(SHARED_ALL_PRIOR_CONTRACT_PATH, model_contract)


Select GAM training rows:   0%|          | 0/752 [00:00<?, ?it/s]

In [15]:
# ==== Strict-cold Evaluation Policy ====
# Cold remains in the complete OOF evaluation universe but never enters a/b
# fitting. 12a produces the category-specific cold OOF score source; 12b/12c
# preserve native model scores for interpretation and replace evaluated scores.
feature_df["native_gam_score"] = feature_df["gam_score"].astype(np.float32)
feature_df["s2q_fallback_score"] = np.nan

if CONDITION_KEY == "s2q":
    feature_df["prediction_source"] = "native_s2q_gam"
    s2q_cold_predictions_df = feature_df.loc[
        feature_df["regime"].eq("cold"),
        [
            "query_id", "candidate_item_id", "candidate_rank", "candidate_score",
            "candidate_score_norm", "is_gt", "gam_score", "cv_fold",
        ],
    ].copy()
    s2q_cold_predictions_df["candidate_method_key"] = candidate_method_key
    s2q_cold_predictions_df["cold_candidate_contract_hash"] = stable_frame_hash(
        s2q_cold_predictions_df,
        ["query_id", "candidate_item_id", "candidate_rank", "candidate_score_norm", "is_gt"],
        ["query_id", "candidate_rank", "candidate_item_id"],
    )
    S2Q_COLD_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
    s2q_cold_predictions_df.to_parquet(S2Q_COLD_PREDICTIONS_PATH, index=False)
    cold_fallback_qc_df = pd.DataFrame([{
        "condition": CONDITION_KEY,
        "n_cold_queries": int(query_meta_df["regime"].eq("cold").sum()),
        "candidate_identity_passed": candidate_identity_after_feature_hash == candidate_identity_before_hash,
        "score_identity_passed": True,
        "fold_identity_passed": True,
        "rank_identity_passed": True,
        "policy": "native_s2q_oof",
    }])
else:
    if not S2Q_COLD_PREDICTIONS_PATH.exists():
        raise FileNotFoundError(
            f"Missing same-category 12a cold OOF predictions: {S2Q_COLD_PREDICTIONS_PATH}"
        )
    fallback = pd.read_parquet(S2Q_COLD_PREDICTIONS_PATH)
    required_fallback_columns = {
        "query_id", "candidate_item_id", "candidate_rank", "candidate_score_norm",
        "is_gt", "gam_score", "cv_fold", "cold_candidate_contract_hash",
    }
    missing_fallback_columns = sorted(required_fallback_columns - set(fallback.columns))
    if missing_fallback_columns:
        raise RuntimeError(f"12a cold prediction artifact is missing columns: {missing_fallback_columns}")
    fallback["query_id"] = fallback["query_id"].astype(str)
    fallback["candidate_item_id"] = fallback["candidate_item_id"].astype(str)
    if fallback.duplicated(["query_id", "candidate_item_id"]).any():
        raise RuntimeError("12a cold prediction artifact contains duplicate query/candidate keys.")
    fallback_contract_values = fallback["cold_candidate_contract_hash"].dropna().astype(str).unique()
    if len(fallback_contract_values) != 1:
        raise RuntimeError("12a cold prediction artifact has an invalid candidate-contract hash.")

    cold_mask = feature_df["regime"].eq("cold")
    current_cold = feature_df.loc[
        cold_mask,
        ["query_id", "candidate_item_id", "candidate_rank", "candidate_score_norm", "is_gt", "cv_fold"],
    ].copy()
    current_cold["query_id"] = current_cold["query_id"].astype(str)
    current_cold["candidate_item_id"] = current_cold["candidate_item_id"].astype(str)
    current_cold_contract_hash = stable_frame_hash(
        current_cold,
        ["query_id", "candidate_item_id", "candidate_rank", "candidate_score_norm", "is_gt"],
        ["query_id", "candidate_rank", "candidate_item_id"],
    )
    if current_cold_contract_hash != fallback_contract_values[0]:
        raise RuntimeError(
            "Strict-cold candidate identity/rank/normalized-score/label differs from 12a; "
            "the S2-Q fallback cannot be applied safely."
        )
    merged_cold = current_cold[["query_id", "candidate_item_id", "cv_fold"]].merge(
        fallback[["query_id", "candidate_item_id", "gam_score", "cv_fold"]].rename(
            columns={"gam_score": "s2q_gam_score", "cv_fold": "s2q_cv_fold"}
        ),
        on=["query_id", "candidate_item_id"],
        how="left",
        validate="one_to_one",
    )
    if merged_cold["s2q_gam_score"].isna().any() or len(merged_cold) != int(cold_mask.sum()):
        raise RuntimeError("12a cold score coverage is incomplete.")
    if not merged_cold["cv_fold"].astype(int).eq(merged_cold["s2q_cv_fold"].astype(int)).all():
        raise RuntimeError("12a and current notebook assign a cold query to different OOF folds.")

    feature_df.loc[cold_mask, "gam_score"] = merged_cold["s2q_gam_score"].to_numpy(dtype=np.float32)
    feature_df.loc[cold_mask, "s2q_fallback_score"] = merged_cold["s2q_gam_score"].to_numpy(dtype=np.float32)
    feature_df.loc[cold_mask, "cv_fold"] = merged_cold["s2q_cv_fold"].to_numpy(dtype=int)
    feature_df.loc[cold_mask, "prediction_source"] = "s2q_cold_oof_fallback"
    cold_fallback_qc_df = pd.DataFrame([{
        "condition": CONDITION_KEY,
        "n_cold_queries": int(query_meta_df["regime"].eq("cold").sum()),
        "candidate_identity_passed": True,
        "score_identity_passed": bool(merged_cold["s2q_gam_score"].notna().all()),
        "fold_identity_passed": True,
        "rank_identity_passed": False,
        "policy": "same_category_s2q_oof_score_and_deterministic_rank",
    }])


## 6. Evaluate Fixed Prefixes and Inspect Additive Effects

In [16]:
# ==== Evaluation ====
def metrics_from_target_rank(target_rank: pd.Series, k: int) -> tuple[pd.Series, pd.Series, pd.Series]:
    hit = target_rank.notna() & target_rank.le(k)
    reciprocal = pd.Series(np.where(hit, 1.0 / target_rank.fillna(np.inf), 0.0), index=target_rank.index)
    ndcg = pd.Series(np.where(hit, 1.0 / np.log2(target_rank.fillna(np.inf) + 1.0), 0.0), index=target_rank.index)
    return hit.astype(float), reciprocal.astype(float), ndcg.astype(float)


def score_depth(frame: pd.DataFrame, depth: int, method_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    subset = frame[frame["candidate_rank"].le(depth)].copy()
    if method_name == BASELINE_RERANK_METHOD:
        subset["rerank_rank"] = subset["candidate_rank"].astype(int)
    else:
        subset = subset.sort_values(
            ["query_id", "gam_score", "candidate_score_norm", "candidate_rank", "candidate_item_id"],
            ascending=[True, False, False, True, True], kind="mergesort",
        )
        subset["rerank_rank"] = subset.groupby("query_id").cumcount().add(1).astype(int)
    target_rank = subset.loc[subset["is_gt"].eq(1), ["query_id", "rerank_rank"]].set_index("query_id")["rerank_rank"]
    query_rows = query_meta_df[["query_id", "case_id", "user_id", "regime", "sampling_bracket"]].copy()
    query_rows["target_rank"] = query_rows["query_id"].map(target_rank)
    query_rows["gt_rank"] = query_rows["target_rank"]
    query_rows["pool_depth"] = int(depth)
    query_rows["rerank_method"] = method_name
    query_rows["gt_in_pool"] = query_rows["target_rank"].notna().astype(int)
    for k in [1, 5, 10, depth]:
        hit, reciprocal, ndcg = metrics_from_target_rank(query_rows["target_rank"], int(k))
        query_rows[f"HitRate@{k}"] = hit
        query_rows[f"MRR@{k}"] = reciprocal
        query_rows[f"NDCG@{k}"] = ndcg
    export_columns = [
        "query_id", "case_id", "user_id", "regime", "candidate_item_id", "candidate_brand_raw",
        "candidate_brand_present", *[c for c in BRAND_HISTORY_FEATURES if c in subset.columns],
        "candidate_rank", "candidate_score", "candidate_score_norm", "is_gt", "gam_score",
        "cv_fold", "prediction_source", "native_gam_score", "s2q_fallback_score", "rerank_rank",
    ]
    export = subset[(subset["rerank_rank"].le(EXPORT_RERANKED_TOPK)) | subset["is_gt"].eq(1)][list(dict.fromkeys(export_columns))].copy()
    export["pool_depth"] = int(depth)
    export["rerank_method"] = method_name
    return query_rows, export


with runtime_step("evaluation_all_depths"):
    per_query_frames, reranked_frames = [], []
    for depth in REPORT_POOL_DEPTHS:
        for method_name in [BASELINE_RERANK_METHOD, GAM_OUTPUT_RERANK_METHOD]:
            query_metrics, reranked = score_depth(feature_df, int(depth), method_name)
            per_query_frames.append(query_metrics)
            if depth == max(REPORT_POOL_DEPTHS):
                reranked_frames.append(reranked)
    per_query_results_df = pd.concat(per_query_frames, ignore_index=True)
    reranked_candidates_df = pd.concat(reranked_frames, ignore_index=True)
    metric_columns = [column for column in per_query_results_df.columns if re.match(r"^(HitRate|MRR|NDCG)@", column)]
    summary_by_pool_depth_df = per_query_results_df.groupby(["pool_depth", "rerank_method"], observed=True)[metric_columns].mean().reset_index()
    summary_by_pool_depth_df["n_queries"] = per_query_results_df.groupby(["pool_depth", "rerank_method"], observed=True)["query_id"].nunique().to_numpy()
    summary_by_regime_pool_depth_df = per_query_results_df.groupby(["pool_depth", "regime", "rerank_method"], observed=True)[metric_columns].mean().reset_index()
    summary_by_regime_pool_depth_df["n_queries"] = per_query_results_df.groupby(["pool_depth", "regime", "rerank_method"], observed=True)["query_id"].nunique().to_numpy()
    report_depth = max(REPORT_POOL_DEPTHS)
    summary_overall_df = summary_by_pool_depth_df[summary_by_pool_depth_df["pool_depth"].eq(report_depth)].reset_index(drop=True)
    summary_by_regime_df = summary_by_regime_pool_depth_df[summary_by_regime_pool_depth_df["pool_depth"].eq(report_depth)].reset_index(drop=True)
    baseline_pool = summary_by_pool_depth_df[summary_by_pool_depth_df["rerank_method"].eq(BASELINE_RERANK_METHOD)].set_index("pool_depth")
    gam_pool = summary_by_pool_depth_df[summary_by_pool_depth_df["rerank_method"].eq(GAM_OUTPUT_RERANK_METHOD)].set_index("pool_depth")
    uplift_by_pool_depth_df = gam_pool[metric_columns].subtract(baseline_pool[metric_columns]).reset_index()
    uplift_by_pool_depth_df["comparison"] = f"{GAM_OUTPUT_RERANK_METHOD} - {BASELINE_RERANK_METHOD}"
    uplift_summary_df = uplift_by_pool_depth_df[uplift_by_pool_depth_df["pool_depth"].eq(report_depth)].reset_index(drop=True)
    baseline_regime = summary_by_regime_pool_depth_df[summary_by_regime_pool_depth_df["rerank_method"].eq(BASELINE_RERANK_METHOD)].set_index(["pool_depth", "regime"])
    gam_regime = summary_by_regime_pool_depth_df[summary_by_regime_pool_depth_df["rerank_method"].eq(GAM_OUTPUT_RERANK_METHOD)].set_index(["pool_depth", "regime"])
    uplift_by_regime_pool_depth_df = gam_regime[metric_columns].subtract(baseline_regime[metric_columns]).reset_index()
    uplift_by_regime_pool_depth_df["comparison"] = f"{GAM_OUTPUT_RERANK_METHOD} - {BASELINE_RERANK_METHOD}"
    uplift_by_regime_df = uplift_by_regime_pool_depth_df[uplift_by_regime_pool_depth_df["pool_depth"].eq(report_depth)].reset_index(drop=True)
    display(summary_overall_df[["rerank_method", "n_queries", "NDCG@1", "NDCG@5", "HitRate@5", "MRR@5"]])

# Metric-relevant rank identity is checked after applying the same deterministic
# sorter at every reporting depth. Cold candidate/score identity above implies
# all candidate ranks; target-rank equality is persisted as the compact QC.
cold_gam_metrics = per_query_results_df[
    per_query_results_df["regime"].eq("cold")
    & per_query_results_df["rerank_method"].eq(GAM_OUTPUT_RERANK_METHOD)
][["query_id", "pool_depth", "target_rank"]].copy()
cold_gam_metrics["query_id"] = cold_gam_metrics["query_id"].astype(str)
if CONDITION_KEY == "s2q":
    cold_gam_metrics.to_parquet(S2Q_COLD_TARGET_RANKS_PATH, index=False)
    cold_rank_identity_passed = True
else:
    if not S2Q_COLD_TARGET_RANKS_PATH.exists():
        raise FileNotFoundError(
            f"Missing 12a cold target-rank artifact: {S2Q_COLD_TARGET_RANKS_PATH}"
        )
    expected_cold_ranks = pd.read_parquet(S2Q_COLD_TARGET_RANKS_PATH)
    require_columns(expected_cold_ranks, ["query_id", "pool_depth", "target_rank"], "12a cold target ranks")
    expected_cold_ranks["query_id"] = expected_cold_ranks["query_id"].astype(str)
    rank_qc = cold_gam_metrics.merge(
        expected_cold_ranks.rename(columns={"target_rank": "s2q_target_rank"}),
        on=["query_id", "pool_depth"], how="outer", validate="one_to_one", indicator=True,
    )
    same_missing = rank_qc["target_rank"].isna().eq(rank_qc["s2q_target_rank"].isna())
    same_present = rank_qc["target_rank"].fillna(-1).eq(rank_qc["s2q_target_rank"].fillna(-1))
    cold_rank_identity_passed = bool(
        rank_qc["_merge"].eq("both").all() and same_missing.all() and same_present.all()
    )
    if not cold_rank_identity_passed:
        raise RuntimeError("Cold RankP/Full target ranks do not exactly reproduce Base at every pool depth.")
cold_fallback_qc_df.loc[:, "rank_identity_passed"] = bool(cold_rank_identity_passed)



,rerank_method,n_queries,NDCG@1,NDCG@5,HitRate@5,MRR@5
0,gam_no_prior_rerank,1968,0.013720,0.028359,0.043191,0.023518
1,stage1_baseline,1968,0.010671,0.023847,0.037093,0.019521


In [17]:
# ==== Interpretable Additive Effects ====
if gam_effect_curves_df.empty:
    gam_effect_summary_df = pd.DataFrame(columns=["feature", "effect_min", "effect_max", "effect_range", "low_to_high_change", "direction"])
    gam_fold_effect_stability_df = pd.DataFrame(columns=["feature", "fold_correlation"])
else:
    effect_fold_summary = (
        gam_effect_curves_df.groupby(["fold", "feature"], observed=True)
        .agg(
            effect_min=("centered_additive_effect", "min"),
            effect_max=("centered_additive_effect", "max"),
            low_effect=("centered_additive_effect", "first"),
            high_effect=("centered_additive_effect", "last"),
        )
        .reset_index()
    )
    effect_fold_summary["effect_range"] = effect_fold_summary["effect_max"] - effect_fold_summary["effect_min"]
    effect_fold_summary["low_to_high_change"] = effect_fold_summary["high_effect"] - effect_fold_summary["low_effect"]

    gam_effect_summary_df = (
        effect_fold_summary.groupby("feature", observed=True)
        .agg(
            effect_min=("effect_min", "mean"),
            effect_max=("effect_max", "mean"),
            effect_range=("effect_range", "mean"),
            low_to_high_change=("low_to_high_change", "mean"),
            fold_effect_range_std=("effect_range", "std"),
        )
        .reset_index()
        .sort_values("effect_range", ascending=False)
        .reset_index(drop=True)
    )
    gam_effect_summary_df["direction"] = np.select(
        [gam_effect_summary_df["low_to_high_change"].gt(1e-6), gam_effect_summary_df["low_to_high_change"].lt(-1e-6)],
        ["increasing", "decreasing"],
        default="flat_or_non_monotonic",
    )

    stability_rows = []
    for feature, feature_rows in gam_effect_curves_df.groupby("feature", sort=False):
        pivot = feature_rows.pivot(index="grid_value", columns="fold", values="centered_additive_effect").sort_index()
        fold_columns = list(pivot.columns)
        correlation = np.nan
        if len(fold_columns) >= 2:
            correlation = float(pivot[fold_columns[0]].corr(pivot[fold_columns[1]]))
        stability_rows.append({"feature": feature, "fold_correlation": correlation})
    gam_fold_effect_stability_df = pd.DataFrame(stability_rows)

feature_importance_df = gam_effect_summary_df.rename(columns={"effect_range": "importance"}).copy()
feature_importance_summary_df = feature_importance_df.copy()

display(gam_effect_summary_df.head(12))

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


,feature,effect_min,effect_max,effect_range,low_to_high_change,fold_effect_range_std,direction
0,item_last_review_gap_days,-0.723991,2.814595,3.538585,-2.022062,0.115508,decreasing
1,candidate_rank_pct,-1.321381,0.466492,1.787873,1.271636,0.119591,increasing
2,item_prequery_review_count_log1p,-1.286718,0.293637,1.580355,1.573462,0.130366,increasing
3,candidate_score_norm,-1.185407,0.028646,1.214054,0.463254,0.123065,increasing
4,query_item_structured_match,-0.402255,0.015448,0.417704,0.315615,0.085192,increasing
5,candidate_brand_present,0.000000,0.000000,0.000000,0.000000,0.000000,flat_or_non_monotonic


## 7. Export Base Outputs and Cold Controls

In [18]:
# ==== Export ====
with runtime_step("export_outputs"):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    output_paths = {
        "results_overall": OUT_DIR / "results_overall.csv",
        "results_by_pool_depth": OUT_DIR / "results_by_pool_depth.csv",
        "results_by_regime": OUT_DIR / "results_by_regime.csv",
        "results_by_regime_pool_depth": OUT_DIR / "results_by_regime_pool_depth.csv",
        "uplift_summary": OUT_DIR / "uplift_summary.csv",
        "uplift_by_pool_depth": OUT_DIR / "uplift_by_pool_depth.csv",
        "uplift_by_regime": OUT_DIR / "uplift_by_regime.csv",
        "uplift_by_regime_pool_depth": OUT_DIR / "uplift_by_regime_pool_depth.csv",
        "per_query_metrics": OUT_DIR / "per_query_metrics.parquet",
        "reranked_candidates": OUT_DIR / "reranked_candidates.parquet",
        "query_profiles": OUT_DIR / "query_profiles.parquet",
        "profile_diagnostics_by_regime": OUT_DIR / "profile_diagnostics_by_regime.csv",
        "feature_importance": OUT_DIR / "feature_importance.csv",
        "feature_importance_summary": OUT_DIR / "feature_importance_summary.csv",
        "gam_effect_curves": OUT_DIR / "gam_effect_curves.csv",
        "gam_effect_summary": OUT_DIR / "gam_effect_summary.csv",
        "gam_fold_effect_stability": OUT_DIR / "gam_fold_effect_stability.csv",
        "gam_model_cv_summary": OUT_DIR / "gam_model_cv_summary.csv",
        "gam_negative_sampling_qc": OUT_DIR / "gam_negative_sampling_qc.csv",
        "cold_fallback_qc": OUT_DIR / "cold_fallback_qc.csv",
        "used_model_features": OUT_DIR / "used_model_features.csv",
        "feature_manifest": OUT_DIR / "feature_manifest.json",
        "feature_table_preview": OUT_DIR / "feature_table_preview.csv",
        "runtime_steps": OUT_DIR / "runtime_steps.csv",
        "runtime_by_pool_depth": OUT_DIR / "runtime_by_pool_depth.csv",
        "config_snapshot": OUT_DIR / "config_snapshot.json",
        "run_manifest": OUT_DIR / "run_manifest.json",
        "diagnostics_summary": OUT_DIR / "diagnostics_summary.csv",
        "feature_leakage_qc": OUT_DIR / "feature_leakage_qc.csv",
        "temporal_feature_summary": OUT_DIR / "temporal_feature_summary.csv",
        "query_history_summary": OUT_DIR / "query_history_summary.csv",
        "global_family_weights": OUT_DIR / "global_family_weights.csv",
        "gam_tuning_grid_summary": OUT_DIR / "gam_tuning_grid_summary.csv",
        "report_summary_scope_qc": OUT_DIR / "report_summary_scope_qc.csv",
        "feature_table": OUT_DIR / "feature_table.parquet",
        "gam_preference_alignment_qc": OUT_DIR / "gam_preference_alignment_qc.csv",
        "runtime_notebook_summary": OUT_DIR / "runtime_notebook_summary.csv",
        "runtime_method_summary_at1000": OUT_DIR / "runtime_method_summary_at1000.csv",
        "runtime_pool_depth_diagnostic": OUT_DIR / "runtime_pool_depth_diagnostic.csv",
        "runtime_method_components_at1000": RUNTIME_METHOD_COMPONENTS_PATH,
        "sample_count_qc": OUT_DIR / "sample_count_qc.csv",
    }
    summary_overall_df.to_csv(output_paths["results_overall"], index=False)
    summary_by_pool_depth_df.to_csv(output_paths["results_by_pool_depth"], index=False)
    summary_by_regime_df.to_csv(output_paths["results_by_regime"], index=False)
    summary_by_regime_pool_depth_df.to_csv(output_paths["results_by_regime_pool_depth"], index=False)
    uplift_summary_df.to_csv(output_paths["uplift_summary"], index=False)
    uplift_by_pool_depth_df.to_csv(output_paths["uplift_by_pool_depth"], index=False)
    uplift_by_regime_df.to_csv(output_paths["uplift_by_regime"], index=False)
    uplift_by_regime_pool_depth_df.to_csv(output_paths["uplift_by_regime_pool_depth"], index=False)
    per_query_results_df.to_parquet(output_paths["per_query_metrics"], index=False)
    reranked_candidates_df.to_parquet(output_paths["reranked_candidates"], index=False)
    query_profiles_df.to_parquet(output_paths["query_profiles"], index=False)
    profile_numeric = [c for c in ["profile_history_event_n", "profile_history_item_n", "prior_event_count_log1p", "prior_unique_item_count_log1p", "history_item_count_log1p", "history_strength", "profile_entropy_mean", "profile_concentration", "user_prior_unique_brand_count", "user_prior_brand_entropy_norm"] if c in query_profiles_df.columns]
    profile_diagnostics_by_regime_df = query_profiles_df.merge(query_meta_df[["query_id", "regime"]], on="query_id", how="left", suffixes=("", "_meta")).groupby("regime", sort=False)[profile_numeric].mean().reset_index()
    profile_diagnostics_by_regime_df.to_csv(output_paths["profile_diagnostics_by_regime"], index=False)
    feature_importance_df.to_csv(output_paths["feature_importance"], index=False)
    feature_importance_summary_df.to_csv(output_paths["feature_importance_summary"], index=False)
    gam_effect_curves_df.to_csv(output_paths["gam_effect_curves"], index=False)
    gam_effect_summary_df.to_csv(output_paths["gam_effect_summary"], index=False)
    gam_fold_effect_stability_df.to_csv(output_paths["gam_fold_effect_stability"], index=False)
    model_cv_summary_df.to_csv(output_paths["gam_model_cv_summary"], index=False)
    negative_sampling_qc_df.to_csv(output_paths["gam_negative_sampling_qc"], index=False)
    cold_fallback_qc_df.to_csv(output_paths["cold_fallback_qc"], index=False)
    used_model_features_df.to_csv(output_paths["used_model_features"], index=False)
    feature_df.head(1000).to_csv(output_paths["feature_table_preview"], index=False)
    if EXPORT_FEATURE_TABLE:
        feature_df.to_parquet(output_paths["feature_table"], index=False)
    contract_diagnostics_df.to_csv(output_paths["diagnostics_summary"], index=False)
    contract_diagnostics_df[contract_diagnostics_df["check"].isin(["raw_review_columns_in_model_input", "brand_terms_added_to_synthetic_query", "same_target_item_prior_rows"])].to_csv(output_paths["feature_leakage_qc"], index=False)
    used_model_features_df.assign(is_temporal_feature=used_model_features_df["feature"].str.contains("recency")).to_csv(output_paths["temporal_feature_summary"], index=False)
    query_profiles_df[[c for c in ["query_id", "profile_history_event_n", "profile_history_item_n", "user_prior_unique_brand_count"] if c in query_profiles_df.columns]].to_csv(output_paths["query_history_summary"], index=False)
    pd.DataFrame({"facet_role": NONBRAND_PROFILE_ROLES, "weight": 1.0}).to_csv(output_paths["global_family_weights"], index=False)
    pd.DataFrame([{"C": GAM_C, "selected": True, "n_knots": GAM_N_KNOTS, "degree": GAM_SPLINE_DEGREE}]).to_csv(output_paths["gam_tuning_grid_summary"], index=False)
    pd.DataFrame([{"scope": "all_configured_pool_depths", "pool_depths": json.dumps(REPORT_POOL_DEPTHS), "check_passed": True}]).to_csv(output_paths["report_summary_scope_qc"], index=False)

    runtime_steps_df = pd.DataFrame(RUNTIME_ROWS)
    runtime_steps_df.to_csv(output_paths["runtime_steps"], index=False)
    runtime_by_pool_depth_df = pd.DataFrame([{
        "pool_depth": int(depth), "fit_pool_depth": FIT_POOL_DEPTH, "fit_reused_from_max_depth": depth != FIT_POOL_DEPTH,
        "feature_preparation_runtime_sec": float(runtime_steps_df.loc[runtime_steps_df["step_name"].str.contains("feature|profile|item", regex=True), "runtime_sec"].sum()),
        "model_fit_and_score_runtime_sec": float(runtime_steps_df.loc[runtime_steps_df["step_name"].str.contains("gam_", regex=True), "runtime_sec"].sum()),
        "evaluation_runtime_sec": float(runtime_steps_df.loc[runtime_steps_df["step_name"].eq("evaluation_all_depths"), "runtime_sec"].sum()),
    } for depth in REPORT_POOL_DEPTHS])
    runtime_by_pool_depth_df.to_csv(output_paths["runtime_by_pool_depth"], index=False)
    pd.DataFrame([{
        "method": GAM_OUTPUT_RERANK_METHOD,
        "total_notebook_runtime_sec": float(time.perf_counter() - NOTEBOOK_TIMER_START),
        "sample_scope": "smoke" if SMOKE_TEST else "full",
    }]).to_csv(output_paths["runtime_notebook_summary"], index=False)
    runtime_by_pool_depth_df.loc[runtime_by_pool_depth_df["pool_depth"].eq(FIT_POOL_DEPTH)].to_csv(output_paths["runtime_method_summary_at1000"], index=False)
    runtime_by_pool_depth_df.to_csv(output_paths["runtime_pool_depth_diagnostic"], index=False)
    runtime_by_pool_depth_df.loc[runtime_by_pool_depth_df["pool_depth"].eq(FIT_POOL_DEPTH)].to_csv(output_paths["runtime_method_components_at1000"], index=False)
    contract_diagnostics_df.to_csv(output_paths["gam_preference_alignment_qc"], index=False)
    negative_sampling_qc_df.to_csv(output_paths["sample_count_qc"], index=False)

    feature_manifest = {
        **FEATURE_CONTRACT,
        "feature_contract_hash": FEATURE_CONTRACT_HASH,
        "specification_role": SPECIFICATION_ROLE,
        "primary_registry_policy_version": PRIMARY_REGISTRY_POLICY_VERSION,
        "exact_item_familiarity_enabled": ENABLE_EXACT_ITEM_FAMILIARITY,
        "brand_query_enabled": False,
        "brand_candidate_visible": "candidate_brand_raw" in feature_df.columns,
        "brand_reranking_enabled": "candidate_brand_present" in MODEL_FEATURES,
        "user_brand_affinity_enabled": bool(user_brand_feature_columns),
        "history_source": "all_prior" if USE_USER_PRIOR_FEATURES else "not_loaded",
        "historical_population_review_signals_in_user_profile": False,
        "candidate_brand_audit_column": "candidate_brand_raw",
        "query_brand_leak_qc_source": query_brand_leak_qc_source,
        "query_brand_leak_row_count": int(brand_terms_added_to_synthetic_query_n),
        "s2q_feature_columns": S2Q_FEATURES,
        "shared_all_prior_feature_columns": SHARED_ALL_PRIOR_FEATURES,
        "feature_columns_S2P_equal_feature_columns_Full": feature_schema_equality_passed,
        "fit_pool_depth": FIT_POOL_DEPTH,
        "report_pool_depths": REPORT_POOL_DEPTHS,
    }
    write_json(output_paths["feature_manifest"], feature_manifest)
    config_snapshot = {
        "notebook_name": NOTEBOOK_NAME, "category_id": CATEGORY_ID, "category_label": CATEGORY_LABEL,
        "condition": CONDITION_KEY, "candidate_pool_role": CANDIDATE_POOL_ROLE,
        "candidate_method_key": candidate_method_key, "candidate_method_label": candidate_method_label,
        "candidate_path": str(candidate_path), "stage1_manifest_path": str(STAGE1_CANDIDATE_MANIFEST_PATH),
        "out_dir": str(OUT_DIR), "shared_all_prior_model_dir": str(SHARED_ALL_PRIOR_MODEL_DIR),
        "shared_all_prior_model_role": SHARED_ALL_PRIOR_MODEL_ROLE,
        "fit_pool_depth": FIT_POOL_DEPTH, "report_pool_depths": REPORT_POOL_DEPTHS,
        "n_folds": GAM_N_FOLDS, "n_knots": GAM_N_KNOTS, "spline_degree": GAM_SPLINE_DEGREE,
        "fixed_c": GAM_C, "max_iter": GAM_MAX_ITER, "tol": GAM_TOL, "class_weight": GAM_CLASS_WEIGHT, "solver": GAM_SOLVER,
        "all_prior_definition": ALL_PRIOR_DEFINITION if USE_USER_PRIOR_FEATURES else "not_loaded",
        "cold_user_policy": "native_s2q" if CONDITION_KEY == "s2q" else "same_category_s2q_oof_score_and_rank",
        "feature_contract_hash": FEATURE_CONTRACT_HASH,
        "specification_role": SPECIFICATION_ROLE,
        "primary_registry_policy_version": PRIMARY_REGISTRY_POLICY_VERSION,
        "exact_item_familiarity_enabled": ENABLE_EXACT_ITEM_FAMILIARITY,
        "query_brand_leak_qc_source": query_brand_leak_qc_source,
        "query_brand_leak_row_count": int(brand_terms_added_to_synthetic_query_n),
    }
    write_json(output_paths["config_snapshot"], config_snapshot)
    run_manifest = {
        **config_snapshot,
        "query_count": int(query_meta_df["query_id"].nunique()), "candidate_rows": int(len(feature_df)), "exact_k": expected_k,
        "query_set_hash": query_set_hash, "stage1_manifest_hash": stage1_manifest_hash,
        "candidate_source_signature": candidate_source_signature,
        "candidate_identity_hash": candidate_identity_after_feature_hash,
        "training_rows_hash": shared_training_rows_hash, "labels_hash": shared_labels_hash, "matched_training_membership_hash": matched_training_membership_hash, "fit_query_membership_hash": fit_query_membership_hash, "sample_weight_hash": sample_weight_hash, "training_rows_used": training_rows_used, "n_cold_training_queries": n_cold_training_queries,
        "brand_query_enabled": False, "brand_candidate_visible": True, "brand_reranking_enabled": "candidate_brand_present" in MODEL_FEATURES,
        "query_brand_leak_qc_source": query_brand_leak_qc_source,
        "query_brand_leak_row_count": int(brand_terms_added_to_synthetic_query_n),
        "user_brand_affinity_enabled": bool(user_brand_feature_columns),
        "history_source": "all_prior" if USE_USER_PRIOR_FEATURES else "not_loaded",
        "historical_population_review_signals_in_user_profile": False,
        "input_paths": {
            "stage1_candidate_pool_export_manifest": str(STAGE1_CANDIDATE_MANIFEST_PATH),
            "candidate_pool": str(candidate_path), "query_cache": str(query_cache_path),
            "item_schema": str(ITEM_SCHEMA_PATH), "items_facets": str(ITEM_FACETS_PATH),
            "prior_history": str(PRIOR_HISTORY_PATH) if USE_USER_PRIOR_FEATURES else "not_loaded",
        },
        "outputs": {key: str(value) for key, value in output_paths.items()},
        "runtime_total_sec": float(time.perf_counter() - NOTEBOOK_TIMER_START),
    }
    write_json(output_paths["run_manifest"], run_manifest)
    print("Output directory:", OUT_DIR)


Output directory: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/gam_no_prior


In [19]:
# ==== Final Validation ====
required_output_keys = [
    "results_overall", "results_by_pool_depth", "results_by_regime", "results_by_regime_pool_depth",
    "per_query_metrics", "reranked_candidates", "feature_manifest", "run_manifest",
    "gam_effect_summary", "gam_model_cv_summary", "runtime_steps", "diagnostics_summary",
]
missing_outputs = [str(output_paths[key]) for key in required_output_keys if not Path(output_paths[key]).exists()]
if missing_outputs:
    raise RuntimeError(f"Missing GAM outputs: {missing_outputs}")
observed_methods = sorted(per_query_results_df["rerank_method"].astype(str).unique())
if observed_methods != sorted([BASELINE_RERANK_METHOD, GAM_OUTPUT_RERANK_METHOD]):
    raise RuntimeError(f"Unexpected rerank methods: {observed_methods}")
if sorted(per_query_results_df["pool_depth"].astype(int).unique()) != sorted(REPORT_POOL_DEPTHS):
    raise RuntimeError("Reported pool depths do not match the configured depths.")
if not contract_diagnostics_df["passed"].all():
    raise RuntimeError("Final feature diagnostics failed.")
if CONDITION_KEY == "full" and not feature_schema_equality_passed:
    raise RuntimeError("Full feature schema differs from RankP.")
print("Final validation: PASS")
print("Candidate method:", candidate_method_key, "-", candidate_method_label)
print("Feature contract hash:", FEATURE_CONTRACT_HASH)
print("Training rows sampled:", int(len(training_sample_df)))
print("Full candidate rows scored:", int(len(feature_df)))

# Batch-1 common-framework invariants.
if GAM_N_FOLDS != 5:
    raise RuntimeError("Final GAM benchmark must use five user-group-disjoint folds.")
if GAM_SOLVER != "lbfgs" or GAM_CLASS_WEIGHT is not None:
    raise RuntimeError("Final GAM solver/weighting contract is not active.")
if n_cold_training_queries != 0:
    raise RuntimeError("Cold queries entered GAM fitting.")
if CONDITION_KEY == "full" and training_rows_used != 0:
    raise RuntimeError("Full retrained a GAM model; expected load-and-score only.")
if CONDITION_KEY in {"s2p", "full"} and not bool(cold_fallback_qc_df["rank_identity_passed"].all()):
    raise RuntimeError("Cold target-rank identity against 12a failed.")
if set(MODEL_FEATURES) & set(QUERY_CONSTANT_DIAGNOSTIC_FEATURES):
    raise RuntimeError("A query-constant diagnostic entered the additive GAM ranking vector.")
if len(S2Q_FEATURES) != EXPECTED_S2Q_FEATURE_COUNT:
    raise RuntimeError("Base does not use the six-feature primary GAM base registry.")
if len(SHARED_ALL_PRIOR_FEATURES) != EXPECTED_ALL_PRIOR_FEATURE_COUNT:
    raise RuntimeError("RankP/Full do not use the fifteen-feature primary GAM registry.")
if set(MODEL_FEATURES) & set(EXACT_ITEM_DIAGNOSTIC_FEATURES):
    raise RuntimeError("Previously-reviewed-item diagnostics entered the primary GAM model.")
if GAM_N_KNOTS != 6 or GAM_SPLINE_DEGREE != 3 or GAM_MAX_ITER != 500:
    raise RuntimeError("Matched GAM spline-capacity contract is not active.")
if GAM_C != 2.0 or GAM_TOL != 1e-4:
    raise RuntimeError("Matched GAM regularization/optimizer contract is not active.")
print("Batch-1 matched GAM common-framework validation: PASS")



Final validation: PASS
Candidate method: hybrid_dense_bm25 - Dense-BM25 Hybrid
Feature contract hash: 34c37b221959366e44fcbb03af63dc3ec51976744ffd809548086d8176033ba8
Training rows sampled: 48880
Full candidate rows scored: 1968000
Batch-1 matched GAM common-framework validation: PASS


In [20]:
# ==== Final Report Package Export (lineage, values, gates) ====
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math

if "OUT_DIR" not in globals():
    raise NameError("OUT_DIR is not available; run the notebook cells above before exporting the report package.")

_REPORT_DIR = Path(OUT_DIR) / "report"
_REPORT_DIR.mkdir(parents=True, exist_ok=True)

_NOT_AVAILABLE = "not_available_in_notebook"
_LIGHT_HASH_SUFFIXES = {".json", ".csv", ".txt", ".yaml", ".yml"}
_MAX_LIGHT_HASH_BYTES = 50 * 1024 * 1024


def _json_clean(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): _json_clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_json_clean(v) for v in value]
    try:
        if hasattr(value, "item"):
            return _json_clean(value.item())
    except Exception:
        pass
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, (str, bool, int)) or value is None:
        return value
    try:
        if hasattr(value, "isoformat"):
            return value.isoformat()
    except Exception:
        pass
    return str(value)


def _safe_records(df, columns=None, max_rows=200):
    if df is None or not hasattr(df, "copy"):
        return None
    frame = df.copy()
    if columns is not None:
        frame = frame[[c for c in columns if c in frame.columns]]
    if max_rows is not None:
        frame = frame.head(max_rows)
    return _json_clean(frame.to_dict(orient="records"))


def _path_str(value):
    if value in (None, ""):
        return None
    try:
        return str(Path(value))
    except Exception:
        return str(value)


def _output_path(key):
    paths = globals().get("output_paths", {})
    if isinstance(paths, dict):
        return _path_str(paths.get(key))
    return None


def _hash_file(path, allow_heavy=False):
    if path in (None, ""):
        return None
    try:
        p = Path(path)
        if not p.exists() or not p.is_file():
            return None
        if not allow_heavy:
            if p.suffix.lower() not in _LIGHT_HASH_SUFFIXES:
                return None
            if p.stat().st_size > _MAX_LIGHT_HASH_BYTES:
                return None
        h = hashlib.sha256()
        with p.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None


def _resolve_notebook_path():
    for name in ("__vsc_ipynb_file__", "NOTEBOOK_PATH", "notebook_path"):
        value = globals().get(name)
        if value:
            try:
                p = Path(value)
                if p.exists():
                    return p
            except Exception:
                pass
    return None


def _collect_input_paths():
    names = [
        "STAGE1_CANDIDATE_MANIFEST_PATH", "CANDIDATE_POOL_PATH", "candidate_path",
        "query_cache_path", "QUERY_CACHE_PARQUET", "ITEM_SCHEMA_PATH", "ITEM_FACETS_PATH",
        "ITEM_SCHEMA_PARQUET", "ITEM_DOCS_PARQUET", "ITEMS_FACETS_PARQUET",
        "RETRIEVAL_ARTIFACT_MANIFEST_PATH", "PRIOR_HISTORY_PATH", "PRIOR_HISTORY_PARQUET",
        "TARGET_CASE_METADATA_PARQUET", "ITEM_REVIEW_TIME_INDEX_PATH",
        "TEMPORAL_ARTIFACT_MANIFEST_PATH", "S2Q_COLD_PREDICTIONS_PATH",
        "S2Q_COLD_TARGET_RANKS_PATH", "S2Q_MODEL_CONTRACT_PATH",
        "SHARED_ALL_PRIOR_SCHEMA_PATH", "SHARED_ALL_PRIOR_CONTRACT_PATH",
        "SHARED_ALL_PRIOR_FOLD_ASSIGNMENTS_PATH", "COMMON_QUERY_FOLD_ASSIGNMENTS_PATH",
        "COMMON_QUERY_FOLD_MANIFEST_PATH", "COMMON_GAM_TRAINING_MEMBERSHIP_PATH",
        "COMMON_GAM_TRAINING_MEMBERSHIP_MANIFEST_PATH", "ITEM_CACHE_PATH",
        "ITEM_CACHE_MANIFEST_PATH", "BASE_FEATURE_CACHE_PATH", "BASE_FEATURE_CACHE_MANIFEST_PATH",
        "PROFILE_CACHE_PATH", "PROFILE_CACHE_MANIFEST_PATH", "PRIOR_FEATURE_CACHE_PATH",
        "PRIOR_FEATURE_CACHE_MANIFEST_PATH",
    ]
    paths = []
    for name in names:
        if name in globals():
            value = globals().get(name)
            if isinstance(value, (str, Path)):
                paths.append(value)
    manifest = globals().get("stage1_manifest")
    if isinstance(manifest, dict):
        for section in ("input_paths", "output_paths"):
            values = manifest.get(section, {})
            if isinstance(values, dict):
                for value in values.values():
                    if isinstance(value, (str, Path)):
                        paths.append(value)
    seen, rows = set(), []
    for value in paths:
        p = _path_str(value)
        if not p or p in seen:
            continue
        seen.add(p)
        rows.append({"path": p, "sha256": _hash_file(p)})
    return rows


def _add_value(rows, claim_id, value, source_file=None, aggregation=None, n=None, ci=None, p=None):
    rows.append({
        "claim_id": claim_id,
        "value": _json_clean(value),
        "ci": _json_clean(ci),
        "p": _json_clean(p),
        "n": _json_clean(n),
        "source_file": source_file,
        "aggregation": aggregation or _NOT_AVAILABLE,
    })


def _missing_value(rows, claim_id, source_file=None):
    _add_value(rows, claim_id, None, source_file=source_file, aggregation=_NOT_AVAILABLE)


def _gam_rows(frame_name, source_key, aggregation_prefix, rows):
    frame = globals().get(frame_name)
    if frame is None or not hasattr(frame, "copy"):
        _missing_value(rows, f"gam_{CONDITION_KEY}_ndcg_at_5_{frame_name}", _output_path(source_key))
        return
    frame = frame.copy()
    if "rerank_method" in frame.columns and "GAM_OUTPUT_RERANK_METHOD" in globals():
        frame = frame[frame["rerank_method"].eq(GAM_OUTPUT_RERANK_METHOD)]
    if "pool_depth" in frame.columns and "FIT_POOL_DEPTH" in globals():
        frame = frame[frame["pool_depth"].astype(int).eq(int(FIT_POOL_DEPTH))]
    if frame.empty or "NDCG@5" not in frame.columns:
        _missing_value(rows, f"gam_{CONDITION_KEY}_ndcg_at_5_{frame_name}", _output_path(source_key))
        return
    label_columns = [c for c in ["pool_depth", "regime"] if c in frame.columns]
    for i, record in enumerate(_safe_records(frame, max_rows=200)):
        suffix = "_".join(str(record.get(c)) for c in label_columns if record.get(c) is not None) or str(i)
        n_value = record.get("n_queries", record.get("n", None))
        _add_value(
            rows,
            f"gam_{CONDITION_KEY}_ndcg_at_5_{frame_name}_{suffix}",
            record.get("NDCG@5"),
            source_file=_output_path(source_key),
            aggregation=f"{aggregation_prefix}; GAM condition at FIT_POOL_DEPTH only",
            n=n_value,
        )


def _gate_self_flag_from_qc(frame, columns):
    if frame is None or not hasattr(frame, "columns"):
        return False
    present = [c for c in columns if c in frame.columns]
    if not present:
        return False
    try:
        return not bool(frame[present].astype(bool).all().all())
    except Exception:
        return False


def _rank_qc_summary():
    rank_qc = globals().get("rank_qc")
    if rank_qc is None or not hasattr(rank_qc, "columns"):
        cold_metrics = globals().get("cold_gam_metrics")
        return {
            "source": "native_s2q_or_rank_qc_not_materialized",
            "pool_depths_observed": _json_clean(sorted(cold_metrics["pool_depth"].dropna().astype(int).unique().tolist())) if cold_metrics is not None and hasattr(cold_metrics, "columns") and "pool_depth" in cold_metrics.columns else None,
        }
    summary = {"rows": int(len(rank_qc))}
    if "pool_depth" in rank_qc.columns:
        summary["pool_depths_observed"] = _json_clean(sorted(rank_qc["pool_depth"].dropna().astype(int).unique().tolist()))
    if "_merge" in rank_qc.columns:
        summary["merge_counts"] = _json_clean(rank_qc["_merge"].astype(str).value_counts(dropna=False).to_dict())
    if {"target_rank", "s2q_target_rank"}.issubset(rank_qc.columns):
        mismatch = ~rank_qc["target_rank"].fillna(-1).eq(rank_qc["s2q_target_rank"].fillna(-1))
        summary["target_rank_mismatch_rows"] = int(mismatch.sum())
    return summary


def _fold_summary():
    qf = globals().get("query_fold_df")
    if qf is None or not hasattr(qf, "columns"):
        return {"n_folds_configured": globals().get("GAM_N_FOLDS", None), "fold_assignments": _NOT_AVAILABLE}
    fold_col = "cv_fold" if "cv_fold" in qf.columns else "fold" if "fold" in qf.columns else None
    out = {"n_folds_configured": globals().get("GAM_N_FOLDS", None)}
    if fold_col:
        out["folds_observed"] = _json_clean(sorted(qf[fold_col].dropna().astype(int).unique().tolist()))
        out["queries_by_fold"] = _json_clean(qf.groupby(fold_col, dropna=False).size().to_dict())
    if "user_id" in qf.columns and fold_col:
        user_fold_counts = qf.groupby("user_id")[fold_col].nunique(dropna=True)
        out["multi_fold_user_count"] = int((user_fold_counts > 1).sum())
    return out


def _feature_schema_evidence():
    condition = globals().get("CONDITION_KEY")
    evidence = {"condition": condition, "feature_contract_hash": globals().get("FEATURE_CONTRACT_HASH")}
    for name in ("feature_columns_S2P", "feature_columns_Full", "MODEL_FEATURES"):
        if name in globals():
            evidence[name] = globals().get(name)
    shared_schema = globals().get("shared_schema")
    if isinstance(shared_schema, dict):
        evidence["shared_schema_feature_contract_hash"] = shared_schema.get("feature_contract_hash")
        evidence["shared_schema_feature_columns"] = shared_schema.get("feature_columns")
    elif condition == "s2q":
        evidence["scope"] = "not_applicable_for_s2q"
    else:
        evidence["shared_schema"] = _NOT_AVAILABLE
    return evidence


def _self_flag_feature_schema():
    if globals().get("CONDITION_KEY") != "full":
        return False
    if "feature_schema_equality_passed" not in globals():
        return False
    return not bool(globals().get("feature_schema_equality_passed"))


def _contract_rows():
    return _safe_records(globals().get("contract_diagnostics_df"), max_rows=200) or _NOT_AVAILABLE


_report_values = []
_gam_rows("summary_overall_df", "results_overall", "mean per-case NDCG@5", _report_values)
_gam_rows("summary_by_regime_df", "results_by_regime", "mean per-case NDCG@5 by regime", _report_values)
_gam_rows("summary_by_regime_pool_depth_df", "results_by_regime_pool_depth", "mean per-case NDCG@5 by regime and pool depth", _report_values)
_add_value(
    _report_values,
    f"gam_{CONDITION_KEY}_spline_capacity_contract",
    {"n_knots": globals().get("GAM_N_KNOTS"), "spline_degree": globals().get("GAM_SPLINE_DEGREE"), "max_iter": globals().get("GAM_MAX_ITER"), "fit_pool_depth": globals().get("FIT_POOL_DEPTH")},
    source_file=_output_path("config_snapshot"),
    aggregation="configured GAM spline-capacity contract values",
)
_add_value(
    _report_values,
    f"gam_{CONDITION_KEY}_regularization_optimizer_contract",
    {"C": globals().get("GAM_C"), "solver": globals().get("GAM_SOLVER"), "penalty": "l2", "tol": globals().get("GAM_TOL"), "class_weight": globals().get("GAM_CLASS_WEIGHT")},
    source_file=_output_path("config_snapshot"),
    aggregation="configured GAM regularization and optimizer contract values",
)
_cv = globals().get("model_cv_summary_df")
if _cv is None or not hasattr(_cv, "columns"):
    _missing_value(_report_values, f"gam_{CONDITION_KEY}_per_fold_info", _output_path("gam_model_cv_summary"))
else:
    fold_columns = [c for c in ["fold", "n_train_queries", "n_train_rows", "n_test_queries", "n_test_rows", "n_cold_training_queries", "n_iter", "reused_checkpoint", "fit_runtime_sec", "score_runtime_sec"] if c in _cv.columns]
    for record in _safe_records(_cv, fold_columns, max_rows=50):
        _add_value(
            _report_values,
            f"gam_{CONDITION_KEY}_fold_{record.get('fold', 'unknown')}_info",
            record,
            source_file=_output_path("gam_model_cv_summary"),
            aggregation="per-fold GAM CV/training-summary row already computed by the notebook",
            n=record.get("n_test_queries", record.get("n_test_rows")),
        )

_cold_qc = globals().get("cold_fallback_qc_df")
_cold_pass_columns = ["candidate_identity_passed", "score_identity_passed", "fold_identity_passed", "rank_identity_passed"]
_cold_qc_records = _safe_records(_cold_qc, [c for c in getattr(_cold_qc, "columns", []) if not c.endswith("_passed")], max_rows=20) or _NOT_AVAILABLE
_fold_observed = _fold_summary()
_qc_gates = [
    {
        "gate_id": "cold_fallback_qc",
        "observed": {
            "qc_rows_without_pass_booleans": _cold_qc_records,
            "candidate_contract_hashes": {
                "before_feature_hash": globals().get("candidate_identity_before_hash"),
                "after_feature_hash": globals().get("candidate_identity_after_feature_hash"),
                "current_cold_contract_hash": globals().get("current_cold_contract_hash"),
                "fallback_contract_values": globals().get("fallback_contract_values"),
            },
            "score_evidence": {"fallback_score_missing_rows": int(globals().get("merged_cold")["s2q_gam_score"].isna().sum()) if globals().get("merged_cold") is not None and "s2q_gam_score" in globals().get("merged_cold").columns else None},
            "fold_evidence": {"fold_mismatch_rows": int((globals().get("merged_cold")["cv_fold"] != globals().get("merged_cold")["s2q_cv_fold"]).sum()) if globals().get("merged_cold") is not None and {"cv_fold", "s2q_cv_fold"}.issubset(globals().get("merged_cold").columns) else None},
            "rank_evidence": _rank_qc_summary(),
        },
        "expected_contract": "Cold fallback uses identical candidates/scores/folds and target-rank identity against 12a at every reported pool depth.",
        "self_flag": _gate_self_flag_from_qc(_cold_qc, _cold_pass_columns),
    },
    {
        "gate_id": "load_only_proof_12c",
        "observed": {"condition": globals().get("CONDITION_KEY"), "shared_all_prior_model_role": globals().get("SHARED_ALL_PRIOR_MODEL_ROLE"), "training_rows_used": globals().get("training_rows_used"), "model_cv_rows": len(_cv) if _cv is not None and hasattr(_cv, "__len__") else None},
        "expected_contract": "For 12c Full, Full retrained a GAM model; expected load-and-score only must not trigger.",
        "self_flag": bool(globals().get("CONDITION_KEY") == "full" and globals().get("training_rows_used", 0) != 0),
    },
    {
        "gate_id": "cold_queries_entered_gam_fitting",
        "observed": {"n_cold_training_queries": globals().get("n_cold_training_queries", None)},
        "expected_contract": "Cold queries entered GAM fitting leakage guard remains zero.",
        "self_flag": bool(globals().get("n_cold_training_queries", 0) != 0),
    },
    {
        "gate_id": "five_user_group_disjoint_folds",
        "observed": _fold_observed,
        "expected_contract": "GAM uses five user-group-disjoint folds.",
        "self_flag": bool(globals().get("GAM_N_FOLDS", None) != 5 or (_fold_observed.get("multi_fold_user_count") or 0) != 0),
    },
    {
        "gate_id": "feature_registry",
        "observed": {"condition": globals().get("CONDITION_KEY"), "model_feature_count": len(globals().get("MODEL_FEATURES", [])) if "MODEL_FEATURES" in globals() else None, "s2q_feature_count": len(globals().get("S2Q_FEATURES", [])) if "S2Q_FEATURES" in globals() else None, "shared_all_prior_feature_count": len(globals().get("SHARED_ALL_PRIOR_FEATURES", [])) if "SHARED_ALL_PRIOR_FEATURES" in globals() else None, "model_features": globals().get("MODEL_FEATURES", _NOT_AVAILABLE), "feature_contract_hash": globals().get("FEATURE_CONTRACT_HASH")},
        "expected_contract": "Base uses the 6-feature registry; RankP and Full use the 15-feature registry.",
        "self_flag": bool(("S2Q_FEATURES" in globals() and "EXPECTED_S2Q_FEATURE_COUNT" in globals() and len(S2Q_FEATURES) != EXPECTED_S2Q_FEATURE_COUNT) or ("SHARED_ALL_PRIOR_FEATURES" in globals() and "EXPECTED_ALL_PRIOR_FEATURE_COUNT" in globals() and len(SHARED_ALL_PRIOR_FEATURES) != EXPECTED_ALL_PRIOR_FEATURE_COUNT)),
    },
    {
        "gate_id": "feature_schema_s2p_equals_full",
        "observed": _feature_schema_evidence(),
        "expected_contract": "Full feature schema equals the RankP feature schema and contract hash.",
        "self_flag": _self_flag_feature_schema(),
    },
    {
        "gate_id": "matched_spline_regularization_optimizer_contract",
        "observed": {"n_knots": globals().get("GAM_N_KNOTS"), "spline_degree": globals().get("GAM_SPLINE_DEGREE"), "max_iter": globals().get("GAM_MAX_ITER"), "C": globals().get("GAM_C"), "tol": globals().get("GAM_TOL"), "solver": globals().get("GAM_SOLVER"), "class_weight": globals().get("GAM_CLASS_WEIGHT"), "model_parameter_hash": globals().get("MODEL_PARAMETER_HASH"), "contract_diagnostics_checks": sorted(contract_diagnostics_df["check"].astype(str).tolist()) if "contract_diagnostics_df" in globals() and hasattr(contract_diagnostics_df, "columns") and "check" in contract_diagnostics_df.columns else _NOT_AVAILABLE},
        "expected_contract": "Matched spline-capacity, regularization, and optimizer contract is active.",
        "self_flag": bool(("contract_diagnostics_df" in globals() and hasattr(contract_diagnostics_df, "columns") and "passed" in contract_diagnostics_df.columns and not contract_diagnostics_df["passed"].astype(bool).all()) or globals().get("GAM_SOLVER") != "lbfgs" or globals().get("GAM_CLASS_WEIGHT") is not None),
    },
]

_notebook_path = _resolve_notebook_path()
_lineage = {
    "notebook": _notebook_path.name if _notebook_path else globals().get("NOTEBOOK_NAME", None),
    "category": globals().get("CATEGORY_ID", globals().get("CATEGORY_LABEL", None)),
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "code_sha": _hash_file(_notebook_path, allow_heavy=True) if _notebook_path else None,
    "inputs": _collect_input_paths(),
}

_lineage_path = _REPORT_DIR / "lineage.json"
_values_path = _REPORT_DIR / "report_values.json"
_gates_path = _REPORT_DIR / "qc_gates.json"
_verification_path = _REPORT_DIR / "verification_report.md"

_lineage_path.write_text(json.dumps(_json_clean(_lineage), indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
_values_path.write_text(json.dumps(_json_clean(_report_values), indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
_gates_path.write_text(json.dumps(_json_clean(_qc_gates), indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


def _md_cell(value):
    text = "" if value is None else str(_json_clean(value))
    return text.replace("|", "\\|").replace("\n", " ")


def _md_table(rows, columns):
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(_md_cell(row.get(c)) for c in columns) + " |")
    return "\n".join(lines)

_served_ids = [f"table_6_3_gam_per_case_source_{globals().get('CATEGORY_ID', 'category')}_{globals().get('CONDITION_KEY', 'condition')}"]
_value_table_rows = [{**row, "thesis_value": ""} for row in _report_values]
_gate_table_rows = [{"gate_id": row["gate_id"], "observed": json.dumps(_json_clean(row["observed"]), ensure_ascii=False), "expected_contract": row["expected_contract"], "self_flag": row["self_flag"]} for row in _qc_gates]
_anomalies = [row["gate_id"] for row in _qc_gates if row.get("self_flag")]
_anomaly_lines = [f"- {gate_id}" for gate_id in _anomalies] if _anomalies else ["- none"]
_md = [
    "# Verification Report",
    "",
    "## Identity&lineage",
    f"- notebook: {_md_cell(_lineage['notebook'])}",
    f"- category: {_md_cell(_lineage['category'])}",
    f"- run_utc: {_md_cell(_lineage['run_utc'])}",
    f"- code_sha: {_md_cell(_lineage['code_sha'])}",
    f"- inputs_recorded: {len(_lineage['inputs'])}",
    "",
    "## Served elements (IDs only, no numbers)",
    *_served_ids,
    "",
    "## Computed headline values",
    _md_table(_value_table_rows, ["claim_id", "value", "thesis_value", "ci", "p", "n", "source_file", "aggregation"]),
    "",
    "## QC gates (raw)",
    _md_table(_gate_table_rows, ["gate_id", "observed", "expected_contract", "self_flag"]),
    "",
    "## Self-detected anomalies",
    *_anomaly_lines,
    "",
]
_verification_path.write_text("\n".join(_md), encoding="utf-8")

for _written_path in (_lineage_path, _values_path, _gates_path, _verification_path):
    print(_written_path)


/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/gam_no_prior/report/lineage.json
/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/gam_no_prior/report/report_values.json
/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/gam_no_prior/report/qc_gates.json
/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage2_nonpersonalized_rerank/gam_no_prior/report/verification_report.md
